# Phase 2 — Legal NER (strict character spans) · Kaggle 2× T4

**Inputs (Add Input):** `prefixlm-run1` (tokenizer) + the outputs/datasets of your four Phase-1 runs
(Prefix-LM, encoder-decoder, arch 4, Bi-Scale extended). Settings: GPU T4 ×2, Internet On, secret `HF_TOKEN`.

**Yours (rule 17.5):** `ner_model.py` (heads, adaptation methods incl. LoRMA, KD student) and every value marked `TODO`.
**Harness:** data, char↔token alignment, windows, training loop, strict-F1 validation, KD plumbing, prediction.json + zip.

Order: cells 1-6 once (data check must show oracle F1 ≈ 1.0 and SMOKE TEST PASSED) → write `ner_model.py` → experiments (cell 9), two at a time.

In [14]:
# 1 — setup
import os, json, glob, shutil, time, subprocess, sys
os.chdir("/kaggle/working")
import torch
NGPU = torch.cuda.device_count(); print("GPUs:", NGPU)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", e)

def find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

GPUs: 2


In [15]:
# 2 — Phase-2 data + tokenizer
from huggingface_hub import hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
P2 = {}
for f in ["Train/fold1-text.json", "Train/fold1-gold.json", "Train/fold2-text.json", "Train/fold2-gold.json",
          "Train/labels.txt", "Test/fold3-text.json"]:
    P2[f.split("/")[-1]] = hf_hub_download(REPO, f"Phase_2_LNER/{f}", repo_type="dataset")
print(json.dumps(P2, indent=1))
os.makedirs("tokenizer", exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    shutil.copy(find(fn)[0], f"tokenizer/{fn}")
print(open(P2["labels.txt"]).read())

{
 "fold1-text.json": "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json",
 "fold1-gold.json": "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json",
 "fold2-text.json": "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-text.json",
 "fold2-gold.json": "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-gold.json",
 "labels.txt": "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt",
 "fold3-text.json": "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/

In [16]:
%%writefile ner_data.py
"""ner_data.py — Phase-2 L-NER data handling."""
import json
import zlib
from collections import Counter, defaultdict

LABELS_DEFAULT = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT",
                  "STAT", "PREC", "DATE", "CASENO"]


# ------------------------------------------------------------------ loading
def read_labels(path):
    labs = []
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if line:
            labs += [x.strip() for x in line.replace(",", " ").split() if x.strip()]
    return labs


def _as_text_map(obj):
    if isinstance(obj, dict):
        return {str(k): (v if isinstance(v, str) else v.get("text", "")) for k, v in obj.items()}
    out = {}
    for r in obj:
        out[str(r.get("id", r.get("doc_id")))] = r["text"]
    return out


def _parse_ent(e):
    if isinstance(e, (list, tuple)):
        s, en, lab = int(e[0]), int(e[1]), str(e[2])
        txt = e[3] if len(e) > 3 else None
    else:
        s = int(e.get("start", e.get("start_char", e.get("begin"))))
        en = int(e.get("end", e.get("end_char")))
        lab = str(e.get("label", e.get("type", e.get("entity"))))
        txt = e.get("text", e.get("entity_text"))
    return s, en, lab, txt


def _as_gold_map(obj):
    if isinstance(obj, dict):
        items = obj.items()
    else:
        items = [(r.get("id", r.get("doc_id")), r.get("entities", r.get("spans", []))) for r in obj]
    return {str(k): [_parse_ent(e) for e in v] for k, v in items}


def load_split(text_path, gold_path=None):
    texts = _as_text_map(json.load(open(text_path, encoding="utf-8")))
    gold = _as_gold_map(json.load(open(gold_path, encoding="utf-8"))) if gold_path else None
    return texts, gold


def detect_end_convention(texts, gold):
    exc = inc = neither = total = 0
    examples = []
    for d, ents in gold.items():
        t = texts.get(d, "")
        for s, e, lab, txt in ents:
            if txt is None:
                continue
            total += 1
            a, b = t[s:e] == txt, t[s:e + 1] == txt
            exc += a; inc += (b and not a)
            if not a and not b:
                neither += 1
                if len(examples) < 5:
                    examples.append((d, s, e, lab, txt[:60], t[s:e + 1][:60]))
    inclusive = inc > exc
    return dict(inclusive=inclusive, exclusive_matches=exc, inclusive_matches=inc, neither=neither,
                with_text=total, examples=examples)


def to_internal(gold, inclusive):
    return {d: [(s, e + 1 if inclusive else e, lab) for s, e, lab, _ in ents] for d, ents in gold.items()}


def from_internal(spans, inclusive):
    return {d: [[s, e - 1 if inclusive else e, lab] for s, e, lab in v] for d, v in spans.items()}


def split_docs(doc_ids, val_frac, seed=0):
    val = [d for d in doc_ids if (zlib.crc32(f"{seed}:{d}".encode()) % 10000) < val_frac * 10000]
    vs = set(val)
    return [d for d in doc_ids if d not in vs], val


# ------------------------------------------------------------------ tokenization
def trim_offsets(text, offsets):
    out = []
    for s, e in offsets:
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        out.append((s, e))
    return out


def tokenize_doc(text, encode):
    ids, offs = encode(text)
    return list(ids), trim_offsets(text, offs)


# ------------------------------------------------------------------ labels
def label_names(labels, scheme):
    if scheme == "IO":
        return ["O"] + [f"I-{l}" for l in labels]
    if scheme == "BIO":
        return ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
    if scheme == "BIOES":
        return ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I", "E", "S")]
    raise ValueError(scheme)


def token_labels(offsets, spans, labels, scheme):
    names = label_names(labels, scheme)
    idx = {n: i for i, n in enumerate(names)}
    tags = [0] * len(offsets)
    owner = [None] * len(offsets)
    skipped = 0
    starts = [s for s, _ in offsets]
    import bisect
    for (S, E, lab) in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):
        if lab not in labels:
            continue
        lo = max(0, bisect.bisect_left(starts, S) - 1)
        toks = []
        j = lo
        while j < len(offsets) and offsets[j][0] < E:
            s, e = offsets[j]
            if e > s and s < E and e > S:
                toks.append(j)
            j += 1
        if not toks:
            continue
        if any(owner[t] is not None for t in toks):
            skipped += 1
            continue

        full = list(range(toks[0], toks[-1] + 1))
        for k, t in enumerate(full):
            owner[t] = (S, E, lab)
            if scheme == "IO":
                name = f"I-{lab}"
            elif scheme == "BIO":
                name = f"{'B' if k == 0 else 'I'}-{lab}"
            else:
                if len(full) == 1:
                    name = f"S-{lab}"
                else:
                    name = f"{'B' if k == 0 else ('E' if k == len(full) - 1 else 'I')}-{lab}"
            tags[t] = idx[name]
    return tags, skipped


def decode_spans(tag_ids, offsets, labels, scheme, mode="lenient"):
    names = label_names(labels, scheme)
    if scheme == "IO":
        mode = "lenient"
    out = []
    cur = None

    def close():
        nonlocal cur
        if cur is not None and cur[2] > cur[1]:
            out.append((cur[1], cur[2], cur[0]))
        cur = None

    prev = None
    for t, tid in enumerate(tag_ids):
        s, e = offsets[t]
        name = names[tid]


        same_char = prev is not None and (s, e) == prev and e > s
        prev = (s, e)
        if same_char and cur is not None:
            continue
        if name == "O":
            close(); continue
        p, lab = name.split("-", 1)
        if e <= s:
            if cur is not None and cur[0] != lab:
                close()
            continue
        if p in ("B", "S") or cur is None or cur[0] != lab:
            if p in ("I", "E") and mode == "strict" and (cur is None or cur[0] != lab):
                close(); continue
            close()
            cur = [lab, s, e]
        else:
            cur[2] = e
        if p in ("E", "S"):
            close()
    close()
    seen, uniq = set(), []
    for x in out:
        if x not in seen:
            seen.add(x); uniq.append(x)
    return uniq


# ------------------------------------------------------------------ windows
def make_windows(n_tokens, window, stride):
    if n_tokens <= window:
        return [0]
    starts = list(range(0, n_tokens - window + 1, stride))
    if starts[-1] + window < n_tokens:
        starts.append(n_tokens - window)
    return starts


# ------------------------------------------------------------------ scoring
def strict_scores(pred, gold, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in set(gold) | set(pred):
        g = set(map(tuple, gold.get(d, [])))
        p = set(map(tuple, pred.get(d, [])))
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = dict(p=P, r=R, f1=F, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    present = [l for l in labels if per[l]["support"] or per[l]["fp"]]
    macro = sum(per[l]["f1"] for l in labels) / len(labels)
    macro_present = sum(per[l]["f1"] for l in present) / max(1, len(present))
    mp = sum(per[l]["p"] for l in labels) / len(labels)
    mr = sum(per[l]["r"] for l in labels) / len(labels)
    return dict(macro_f1=macro, macro_f1_present=macro_present, macro_p=mp, macro_r=mr, per_label=per)


def format_scores(sc):
    lines = [f"strict macro-F1 {sc['macro_f1']:.4f} (over labels present: {sc['macro_f1_present']:.4f}) | "
             f"macro-P {sc['macro_p']:.4f} | macro-R {sc['macro_r']:.4f}"]
    for l, v in sc["per_label"].items():
        lines.append(f"  {l:10s} F1 {v['f1']:.3f}  P {v['p']:.3f}  R {v['r']:.3f}  "
                     f"(tp {v['tp']}, fp {v['fp']}, fn {v['fn']}, support {v['support']})")
    return "\n".join(lines)


# ------------------------------------------------------------------ checks + stats
def oracle_check(texts, spans, encode, labels, scheme, mode="lenient"):
    pred, skipped, mid_token = {}, 0, Counter()
    for d, sp in spans.items():
        ids, offs = tokenize_doc(texts[d], encode)
        tags, sk = token_labels(offs, sp, labels, scheme)
        skipped += sk
        pred[d] = decode_spans(tags, offs, labels, scheme, mode)
        starts = {s for s, e in offs if e > s}
        ends = {e for s, e in offs if e > s}
        for s, e, lab in sp:
            if s not in starts or e not in ends:
                mid_token[lab] += 1
    sc = strict_scores(pred, spans, labels)
    return sc, skipped, mid_token


def dataset_stats(texts, spans, encode, labels):
    lens, n_tok = [], 0
    for d, t in texts.items():
        ids, _ = encode(t)
        lens.append(len(ids)); n_tok += len(ids)
    lens.sort()
    cnt = Counter(lab for v in spans.values() for _, _, lab in v) if spans else Counter()
    overl = 0
    if spans:
        for v in spans.values():
            v = sorted(v)
            for a, b in zip(v, v[1:]):
                if b[0] < a[1]:
                    overl += 1
    unknown = sorted({lab for v in (spans or {}).values() for _, _, lab in v} - set(labels))
    return dict(docs=len(texts), tokens=n_tok,
                doc_tokens_p50=lens[len(lens) // 2] if lens else 0,
                doc_tokens_p90=lens[int(len(lens) * 0.9)] if lens else 0,
                doc_tokens_max=lens[-1] if lens else 0,
                entities=sum(cnt.values()), per_label=dict(cnt.most_common()),
                overlapping_pairs=overl, unknown_labels=unknown)


# ------------------------------------------------------------------ submission
def validate_prediction(pred, texts, labels):
    missing = set(texts) - set(pred)
    extra = set(pred) - set(texts)
    assert not missing, f"{len(missing)} test docs missing, e.g. {list(missing)[:3]}"
    assert not extra, f"{len(extra)} unknown doc ids, e.g. {list(extra)[:3]}"
    n = 0
    for d, v in pred.items():
        assert isinstance(v, list), d
        seen = set()
        for x in v:
            assert isinstance(x, list) and len(x) in (3, 4), (d, x)
            s, e, lab = x[0], x[1], x[2]
            assert isinstance(s, int) and isinstance(e, int) and 0 <= s <= e <= len(texts[d]), (d, x)
            assert lab in labels, (d, x)
            assert (s, e, lab) not in seen, f"duplicate {x} in {d}"
            seen.add((s, e, lab)); n += 1
    return n

Overwriting ner_data.py


In [17]:
%%writefile test_ner_data.py
"""Synthetic tests for ner_data.py (run: python test_ner_data.py). A mock tokenizer imitates ByteLevel BPE:"""
import json, os, random, re, tempfile

import ner_data as nd

PIECE = re.compile(r" ?[A-Za-zÀ-ɏ]+| ?[0-9]+| ?[^\sA-Za-z0-9À-ɏ]+|\s+(?!\S)|\s+")


def mock_encode(text, piece=3):
    ids, offs = [], []
    for m in PIECE.finditer(text):
        s, e = m.span()
        tok = text[s:e]
        lead = 1 if tok.startswith(" ") and len(tok) > 1 and not tok.isspace() else 0
        body_s = s + lead
        if tok.isspace() or (e - body_s) <= piece:
            ids.append(hash(tok) % 1000); offs.append((s, e)); continue
        first = True
        k = body_s
        while k < e:
            ke = min(e, k + piece)
            ids.append(hash(text[k:ke]) % 1000)
            offs.append((s if first else k, ke)); first = False
            k = ke
    return ids, offs


ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name} {detail}")


LAB = nd.LABELS_DEFAULT
doc = ("IN THE SUPREME COURT OF INDIA\n\nCivil Appeal No. 1234 of 2019\n\nRamesh  Kumar Sharma ...Appellant\n"
       "versus\nState of Uttar Pradesh ...Respondent\n\nJudgment dated 12.03.2019 by Justice A.K. Sikri, "
       "under Section 302 of the Indian Penal Code, 1860, relying on (1997) 6 SCC 241. Mr. José Ávila, Advocate, "
       "for the appellant. ₹ 5,00,000 compensation.")


def span(sub, lab, nth=0):
    i = -1
    for _ in range(nth + 1):
        i = doc.index(sub, i + 1)
    return (i, i + len(sub), lab)


ents = [span("SUPREME COURT OF INDIA", "COURT"), span("1234 of 2019", "CASENO"),
        span("Ramesh  Kumar Sharma", "APP"), span("State of Uttar Pradesh", "RESP"),
        span("12.03.2019", "DATE"), span("A.K. Sikri", "JUDGE"),
        span("Section 302 of the Indian Penal Code, 1860", "STAT"), span("(1997) 6 SCC 241", "PREC"),
        span("José Ávila", "A.COUNSEL")]
for s, e, l in ents:
    assert doc[s:e].strip() == doc[s:e]


ids, offs = nd.tokenize_doc(doc, mock_encode)
check("1 trimmed offsets have no edge spaces", all(e == s or (not doc[s].isspace() and not doc[e - 1].isspace())
                                                   for s, e in offs))
raw = mock_encode(doc)[1]
check("  (mock really has leading-space raw offsets)", any(doc[s] == " " and e - s > 1 for s, e in raw))


for scheme in ("IO", "BIO", "BIOES"):
    for mode in ("lenient", "strict"):
        sc, sk, mid = nd.oracle_check({"d": doc}, {"d": ents}, mock_encode, LAB, scheme, mode)
        exp = 1.0
        check(f"2 oracle {scheme}/{mode}", abs(sc["macro_f1_present"] - exp) < 1e-9,
              f"macro_present {sc['macro_f1_present']:.3f} skipped {sk} mid-token {dict(mid)}")


adj = [(doc.index("Kumar"), doc.index("Kumar") + 5, "APP"), (doc.index("Sharma"), doc.index("Sharma") + 6, "APP")]
sc_io = nd.oracle_check({"d": doc}, {"d": adj}, mock_encode, LAB, "IO")[0]
sc_bio = nd.oracle_check({"d": doc}, {"d": adj}, mock_encode, LAB, "BIO")[0]
check("3 adjacent same-label: BIO exact, IO merges", sc_bio["per_label"]["APP"]["f1"] == 1.0 and
      sc_io["per_label"]["APP"]["f1"] < 1.0, f"(BIO {sc_bio['per_label']['APP']['f1']:.2f}, IO {sc_io['per_label']['APP']['f1']:.2f})")


i = doc.index("Sharma")
midg = [(i, i + 2, "APP")]
sc, sk, mid = nd.oracle_check({"d": doc}, {"d": midg}, mock_encode, LAB, "BIO")
check("4 mid-token boundary detected", mid["APP"] == 1 and sc["per_label"]["APP"]["f1"] == 0.0)


ov = [span("State of Uttar Pradesh", "RESP"), span("Uttar Pradesh", "AUTH")]
tags, sk = nd.token_labels(offs, ov, LAB, "BIO")
check("5 overlapping entity skipped + counted", sk == 1)


g_exc = {"d": [(s, e, l, doc[s:e]) for s, e, l in ents]}
g_inc = {"d": [(s, e - 1, l, doc[s:e]) for s, e, l in ents]}
c1, c2 = nd.detect_end_convention({"d": doc}, g_exc), nd.detect_end_convention({"d": doc}, g_inc)
check("6 exclusive detected", c1["inclusive"] is False and c1["neither"] == 0)
check("6 inclusive detected", c2["inclusive"] is True and c2["neither"] == 0)
back = nd.from_internal(nd.to_internal(g_inc, True), True)
check("6 inclusive round trip", [tuple(x) for x in back["d"]] == [(s, e, l) for s, e, l, _ in g_inc["d"]])


good = True
for n in [1, 5, 99, 100, 101, 250, 1000, 1023]:
    for Wn, St in [(100, 50), (100, 100), (64, 48)]:
        st = nd.make_windows(n, Wn, St)
        cov = set()
        for s in st:
            cov.update(range(s, min(n, s + Wn)))
        good &= cov == set(range(n)) and all(s + Wn <= max(n, Wn) for s in st)
check("7 windows cover all tokens", good)


gold = {"a": [(0, 5, "DATE"), (10, 20, "COURT")], "b": [(3, 8, "DATE")]}
pred = {"a": [(0, 5, "DATE"), (10, 19, "COURT")], "b": [(3, 8, "COURT")]}
sc = nd.strict_scores(pred, gold, LAB)
check("8 scorer exact-match", sc["per_label"]["DATE"]["tp"] == 1 and sc["per_label"]["DATE"]["fn"] == 1 and
      sc["per_label"]["COURT"]["fp"] == 2 and sc["per_label"]["COURT"]["tp"] == 0)


random.seed(0)
bad = 0
for _ in range(300):
    names = nd.label_names(LAB, "BIOES")
    tg = [random.randrange(len(names)) if random.random() < 0.4 else 0 for _ in offs]
    for mode in ("lenient", "strict"):
        for s, e, l in nd.decode_spans(tg, offs, LAB, "BIOES", mode):
            if not (0 <= s < e <= len(doc)) or doc[s].isspace() or doc[e - 1].isspace():
                bad += 1
check("9 decoded spans always clean", bad == 0, f"(bad {bad})")


tmp = tempfile.mkdtemp()
json.dump({"d": doc}, open(os.path.join(tmp, "t.json"), "w"))
json.dump({"d": [[s, e, l, doc[s:e]] for s, e, l in ents]}, open(os.path.join(tmp, "g1.json"), "w"))
json.dump([{"id": "d", "entities": [{"start": s, "end": e, "label": l, "text": doc[s:e]} for s, e, l in ents]}],
          open(os.path.join(tmp, "g2.json"), "w"))
t1, g1 = nd.load_split(os.path.join(tmp, "t.json"), os.path.join(tmp, "g1.json"))
t2, g2 = nd.load_split(os.path.join(tmp, "t.json"), os.path.join(tmp, "g2.json"))
check("10 both gold layouts load identically", g1 == g2 and len(g1["d"]) == len(ents))


p = nd.from_internal({"d": [tuple(x) for x in ents]}, False)
check("11 validator accepts good file", nd.validate_prediction(p, {"d": doc}, LAB) == len(ents))
try:
    nd.validate_prediction({"d": [[0, 5, "FOO"]]}, {"d": doc}, LAB); check("11 validator rejects bad label", False)
except AssertionError:
    check("11 validator rejects bad label", True)


ids_ = [str(i) for i in range(200)]
a1, b1 = nd.split_docs(ids_, 0.15, 0); a2, b2 = nd.split_docs(ids_, 0.15, 0)
check("12 split deterministic + disjoint", a1 == a2 and b1 == b2 and not set(a1) & set(b1) and 10 < len(b1) < 50,
      f"(val {len(b1)} of 200)")


txt = "Judge \u00c9mile spoke"
offs13 = [(0, 5), (6, 7), (6, 7), (7, 11), (12, 17)]
names = nd.label_names(LAB, "BIO"); B_J, I_J = names.index("B-JUDGE"), names.index("I-JUDGE")
for tags13, label in [([0, B_J, B_J, I_J, 0], "B,B,I"), ([0, B_J, B_J, 0, 0], "B,B"), ([0, B_J, I_J, I_J, 0], "B,I,I")]:
    out = nd.decode_spans(tags13, offs13, LAB, "BIO", "lenient")
    check(f"13 same-char tokens ({label}) -> one span, no duplicates", len(out) == 1 and len(set(out)) == len(out), str(out))
tg, _ = nd.token_labels(offs13, [(6, 11, "JUDGE")], LAB, "BIO")
check("13 gold over a split character round-trips", nd.decode_spans(tg, offs13, LAB, "BIO") == [(6, 11, "JUDGE")])

print("\nALL TESTS PASSED" if ok else "\nSOME TESTS FAILED")

Overwriting test_ner_data.py


In [18]:
%%writefile ner_train.py
"""ner_train.py — Phase-2 L-NER fine-tuning harness."""
import importlib, json, math, os, random, sys, time, zipfile

import numpy as np
import torch
import torch.nn.functional as F

import ner_data as nd

CFG = json.load(open(sys.argv[1]))
D, TR, MODEL_CFG = CFG["data"], CFG["train"], CFG["model"]
KD = CFG.get("kd") or {}
OUT = CFG["out_dir"]; NAME = CFG["run_name"]
os.makedirs(OUT, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = int(TR.get("seed", 0))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


def log(*a):
    print(*a, flush=True)


# ------------------------------------------------------------------ data
from tokenizers import ByteLevelBPETokenizer
TOK = ByteLevelBPETokenizer(vocab=os.path.join(D["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(D["tokenizer_dir"], "merges.txt"))
PAD_ID = TOK.token_to_id("<pad>")
PAD_ID = 0 if PAD_ID is None else PAD_ID


def encode(text):
    e = TOK.encode(text)
    return e.ids, e.offsets


LABELS = nd.read_labels(D["labels"]) if D.get("labels") else nd.LABELS_DEFAULT
SCHEME, MODE = D["scheme"], D.get("decode", "lenient")
TAGS = nd.label_names(LABELS, SCHEME)
C = len(TAGS)
W, STRIDE = int(D["window"]), int(D["stride"])

texts, gold_raw = {}, {}
for tp, gp in D["train_files"]:
    t, g = nd.load_split(tp, gp)
    texts.update(t); gold_raw.update(g)
conv = nd.detect_end_convention(texts, gold_raw)
INCL = conv["inclusive"]
gold = nd.to_internal(gold_raw, INCL)
test_texts = nd.load_split(D["test_file"])[0] if D.get("test_file") else {}
log(f"labels {len(LABELS)} | scheme {SCHEME} -> {C} tags | gold end is {'INCLUSIVE' if INCL else 'exclusive'} "
    f"({conv['exclusive_matches']} exclusive / {conv['inclusive_matches']} inclusive / {conv['neither']} neither)")

doc_ids = sorted(texts)
if float(D.get("val_frac", 0)) > 0:
    tr_ids, va_ids = nd.split_docs(doc_ids, float(D["val_frac"]), int(D.get("split_seed", 0)))
else:
    tr_ids, va_ids = doc_ids, []
log(f"train docs {len(tr_ids)} | val docs {len(va_ids)} | test docs {len(test_texts)}")

TOKD = {}


def tok(d, src):
    key = (id(src), d)
    if key not in TOKD:
        TOKD[key] = nd.tokenize_doc(src[d], encode)
    return TOKD[key]


# ---- teacher logits for KD (per doc, per token)
TEACH = None
if KD.get("teacher_logits"):
    z = np.load(KD["teacher_logits"])
    meta = json.loads(str(z["__meta__"]))
    assert meta["tags"] == TAGS, "teacher was trained with a different label scheme"
    TEACH = {k[2:]: z[k] for k in z.files if k.startswith("d:")}
    log(f"KD: teacher logits for {len(TEACH)} docs, T={KD['temperature']} alpha={KD['alpha']}")


def build_windows(ids_list, with_tags):
    out, skipped = [], 0
    for d in ids_list:
        ids, offs = tok(d, texts if with_tags else test_texts)
        tags = None
        if with_tags:
            tags, sk = nd.token_labels(offs, gold.get(d, []), LABELS, SCHEME)
            skipped += sk
        for s in nd.make_windows(len(ids), W, STRIDE):
            out.append((d, s, ids[s:s + W], tags[s:s + W] if tags else None))
    return out, skipped


def collate(batch, with_tags=True, with_teacher=False):
    B = len(batch)
    x = torch.full((B, W), PAD_ID, dtype=torch.long)
    m = torch.zeros((B, W), dtype=torch.bool)
    y = torch.full((B, W), -100, dtype=torch.long)
    tch = torch.zeros((B, W, C), dtype=torch.float32) if (with_teacher and TEACH is not None) else None
    for i, (d, s, ids, tags) in enumerate(batch):
        n = len(ids)
        x[i, :n] = torch.tensor(ids); m[i, :n] = True
        if with_tags and tags is not None:
            y[i, :n] = torch.tensor(tags)
        if tch is not None:
            tch[i, :n] = torch.from_numpy(TEACH[d][s:s + n].astype(np.float32))
    return x, m, y, tch


# ------------------------------------------------------------------ model
model_lib = importlib.import_module(CFG.get("model_module", "ner_model"))
model = model_lib.build_ner_model(dict(MODEL_CFG), C).to(DEV)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"parameters: total {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M ({100*n_train/max(1,n_total):.2f}%)")
assert n_train > 0, "no trainable parameters"


@torch.no_grad()
def predict_logits(doc_list, src):
    model.eval()
    out = {}
    wins = []
    for d in doc_list:
        ids, _ = tok(d, src)
        out[d] = (np.zeros((len(ids), C), np.float32), np.zeros(len(ids), np.float32))
        for s in nd.make_windows(len(ids), W, STRIDE):
            wins.append((d, s, ids[s:s + W], None))
    bs = int(TR.get("eval_batch_size", TR["batch_size"]))
    for i in range(0, len(wins), bs):
        b = wins[i:i + bs]
        x, m, _, _ = collate(b, with_tags=False)
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
            lg = model(x.to(DEV), m.to(DEV)).float().cpu().numpy()
        for j, (d, s, ids, _) in enumerate(b):
            n = len(ids)
            out[d][0][s:s + n] += lg[j, :n]; out[d][1][s:s + n] += 1
    model.train()
    return {d: a / np.maximum(c, 1)[:, None] for d, (a, c) in out.items()}


def spans_from_logits(lg, src):
    res = {}
    for d, L in lg.items():
        _, offs = tok(d, src)
        res[d] = nd.decode_spans(L.argmax(-1).tolist(), offs, LABELS, SCHEME, MODE)
    return res


# ------------------------------------------------------------------ training
train_w, skipped = build_windows(tr_ids, True)
log(f"train windows {len(train_w)} (window {W}, stride {STRIDE}) | overlapping gold entities not representable: {skipped}")
bs, acc = int(TR["batch_size"]), int(TR.get("grad_accum", 1))
epochs = int(TR["epochs"])
steps_per_epoch = math.ceil(len(train_w) / (bs * acc))
total_steps = steps_per_epoch * epochs
warm = int(float(TR.get("warmup_frac", 0)) * total_steps)
base_lr = float(TR["lr"])
groups = model.param_groups(base_lr) if hasattr(model, "param_groups") else \
    [{"params": [p for p in model.parameters() if p.requires_grad]}]
opt = torch.optim.AdamW(groups, lr=base_lr, weight_decay=float(TR.get("weight_decay", 0.0)))
for g in opt.param_groups:
    g.setdefault("initial_lr", g["lr"])
scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
cw = TR.get("class_weights")
cw = torch.tensor(cw, dtype=torch.float32, device=DEV) if cw is not None else None
if cw is not None:
    assert len(cw) == C, f"class_weights needs {C} values, one per tag {TAGS}"


def lr_mult(step):
    if step < warm:
        return (step + 1) / max(1, warm)
    sch = TR.get("schedule", "linear")
    t = (step - warm) / max(1, total_steps - warm)
    return 1.0 if sch == "constant" else max(0.0, 1 - t) if sch == "linear" else 0.5 * (1 + math.cos(math.pi * t))


best, best_state, history = -1.0, None, []
step, t0 = 0, time.time()
max_min = float(TR.get("max_minutes", 1e9))
if DEV.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
model.train()
for ep in range(1, epochs + 1):
    random.shuffle(train_w)
    run_loss, nb = 0.0, 0
    for i in range(0, len(train_w), bs):
        x, m, y, tch = collate(train_w[i:i + bs], with_teacher=True)
        x, m, y = x.to(DEV), m.to(DEV), y.to(DEV)
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
            lg = model(x, m)
        lg = lg.float()
        loss = F.cross_entropy(lg.reshape(-1, C), y.reshape(-1), weight=cw, ignore_index=-100)
        if tch is not None:
            T, a = float(KD["temperature"]), float(KD["alpha"])
            v = (y != -100) if KD.get("only_labelled", True) else m
            kl = F.kl_div(F.log_softmax(lg[v] / T, -1), F.softmax(tch.to(DEV)[v] / T, -1), reduction="batchmean")
            loss = (1 - a) * loss + a * T * T * kl
        scaler.scale(loss / acc).backward()
        nb += 1; run_loss += loss.item()
        if nb % acc == 0:
            for g in opt.param_groups:
                g["lr"] = g["initial_lr"] * lr_mult(step)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],
                                           float(TR.get("grad_clip", 1.0)))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        if (time.time() - t0) / 60 > max_min:
            break
    rec = dict(epoch=ep, train_loss=run_loss / max(1, nb), minutes=(time.time() - t0) / 60)
    if va_ids:
        vs = nd.strict_scores(spans_from_logits(predict_logits(va_ids, texts), texts),
                              {d: gold.get(d, []) for d in va_ids}, LABELS)
        rec.update(val_macro_f1=vs["macro_f1"], val_macro_p=vs["macro_p"], val_macro_r=vs["macro_r"])
        score = vs["macro_f1"]
    else:
        score = ep
    history.append(rec)
    log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
    if score > best:
        best = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        log(f"  best so far (epoch {ep})")
    if (time.time() - t0) / 60 > max_min:
        log(f"time limit {max_min} min reached"); break
train_min = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEV.type == "cuda" else 0.0


cur = dict(model.named_parameters())
with torch.no_grad():
    for k, v in best_state.items():
        cur[k].copy_(v.to(DEV))
torch.save(dict(trained=best_state, cfg=CFG, tags=TAGS), os.path.join(OUT, f"{NAME}_trained_params.pt"))

result = dict(run=NAME, tags=TAGS, scheme=SCHEME, decode=MODE, window=W, stride=STRIDE,
              params_total=n_total, params_trainable=n_train, train_minutes=train_min, peak_mem_gb=peak_gb,
              history=history, gold_end_inclusive=INCL)
if va_ids:
    t1 = time.time()
    vl = predict_logits(va_ids, texts)
    vsp = spans_from_logits(vl, texts)
    vs = nd.strict_scores(vsp, {d: gold.get(d, []) for d in va_ids}, LABELS)
    result.update(val=vs, val_infer_seconds=time.time() - t1)
    log("BEST checkpoint on validation:\n" + nd.format_scores(vs))
    json.dump(nd.from_internal(vsp, INCL), open(os.path.join(OUT, f"{NAME}_val_prediction.json"), "w"))

# ---- teacher dump for knowledge distillation (all training docs, best weights)
if CFG.get("dump_teacher_logits"):
    tl = predict_logits(doc_ids, texts)
    np.savez_compressed(os.path.join(OUT, f"{NAME}_teacher_logits.npz"),
                        __meta__=json.dumps(dict(tags=TAGS, run=NAME, window=W, stride=STRIDE)),
                        **{f"d:{d}": v.astype(np.float16) for d, v in tl.items()})
    log(f"teacher logits written for {len(tl)} docs")

# ---- test set -> prediction.json + submission zip
if test_texts:
    t1 = time.time()
    tsp = spans_from_logits(predict_logits(sorted(test_texts), test_texts), test_texts)
    pred = nd.from_internal(tsp, INCL)
    n_ent = nd.validate_prediction(pred, test_texts, LABELS)
    pj = os.path.join(OUT, f"{NAME}_prediction.json")
    json.dump(pred, open(pj, "w", encoding="utf-8"), ensure_ascii=False)
    zp = os.path.join(OUT, f"{NAME}_submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pj, arcname="prediction.json")
    result.update(test_entities=n_ent, test_infer_seconds=time.time() - t1)
    from collections import Counter
    log(f"test: {n_ent} entities in {len(pred)} docs | per label {dict(Counter(x[2] for v in pred.values() for x in v))}")
    log(f"wrote {zp}")
json.dump(result, open(os.path.join(OUT, f"{NAME}_result.json"), "w"), indent=1)
log(f"done: {NAME} | best val macro-F1 {best:.4f} | train {train_min:.1f} min | peak {peak_gb:.1f} GB | "
    f"trainable {n_train/1e6:.3f}M of {n_total/1e6:.2f}M")

Overwriting ner_train.py


In [19]:
%%writefile phase1_loader.py
"""phase1_loader.py — load a Phase-1 model with its trained weights."""
import importlib, json, os

import torch

REGISTRY = os.environ.get("PHASE1_REGISTRY", "phase1_registry.json")


def load_phase1(name, map_location="cpu"):
    reg = json.load(open(REGISTRY))
    assert name in reg, f"{name!r} not in {REGISTRY}: {sorted(reg)}"
    r = reg[name]
    mod = importlib.import_module(r["module"])
    cfg = dict(r["model_cfg"])
    model = mod.build_model(cfg)
    ck = torch.load(r["ckpt"], map_location=map_location, weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, f"{name}: missing {missing[:5]} unexpected {unexpected[:5]}"
    val = ck.get("val_loss") if isinstance(ck, dict) else None
    return model, dict(model_cfg=cfg, ckpt=r["ckpt"], module=r["module"], val_loss=r.get("val_loss", val))

Overwriting phase1_loader.py


In [20]:
%%writefile ner_dummy.py
"""ner_dummy.py — PIPELINE TEST ONLY (smoke test): each token's tag from its own embedding. Never submit it."""
import torch.nn as nn


class _Dummy(nn.Module):
    def __init__(self, vocab, n):
        super().__init__()
        self.emb = nn.Embedding(vocab, 64)
        self.out = nn.Linear(64, n)

    def forward(self, input_ids, attention_mask):
        return self.out(self.emb(input_ids))


def build_ner_model(model_cfg, num_labels):
    return _Dummy(int(model_cfg.get("vocab_size", 32000)), num_labels)

Overwriting ner_dummy.py


In [21]:
%%writefile p1_biscale.py
"""model.py for the Bi-Scale generation notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================== copied from the Bi-Scale training notebook ==============================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, seq_len):
        if seq_len > self.cos_cached.shape[2]:
            self._build_cache(seq_len)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class BiScaleGQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout, ctx_len, rope_theta, is_local=False, window_size=256):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_head     = d_model // n_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.d_head, max_seq=ctx_len, theta=rope_theta)

        causal = torch.tril(torch.ones(ctx_len, ctx_len))
        if is_local:
            causal = causal * torch.triu(torch.ones(ctx_len, ctx_len), diagonal=-window_size)
        self.register_buffer("mask", causal.view(1, 1, ctx_len, ctx_len))


        for proj in [self.q_proj, self.k_proj, self.v_proj, self.o_proj]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, C = x.shape
        dh = self.d_head

        q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)

        cos, sin = self.rope(T)
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)

        scale = dh ** -0.5
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))

        out = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        for proj in [self.gate, self.up, self.down]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LearnedGatedResidual(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Linear(dim, dim, bias=True)

        nn.init.normal_(self.gate.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.gate.bias, 2.0)

    def forward(self, x, sublayer):
        g = torch.sigmoid(self.gate(x))
        return x + g * sublayer

class BiScaleLegalBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        is_local = (layer_idx < (cfg["n_layers"] // 2))
        self.norm1 = RMSNorm(cfg["d_model"])
        self.attn  = BiScaleGQA(
            cfg["d_model"], cfg["n_heads"], cfg["n_kv_heads"],
            cfg["dropout"], cfg["ctx_len"], cfg["rope_theta"],
            is_local=is_local, window_size=cfg["local_window"]
        )
        self.res1 = LearnedGatedResidual(cfg["d_model"])
        self.norm2 = RMSNorm(cfg["d_model"])
        self.ffn   = SwiGLU(cfg["d_model"], cfg["d_ff"])
        self.res2  = LearnedGatedResidual(cfg["d_model"])
        self.drop  = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = self.res1(x, self.drop(self.attn(self.norm1(x))))
        x = self.res2(x, self.drop(self.ffn(self.norm2(x))))
        return x

class BiScaleLegalLLaMA(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.ctx     = cfg["ctx_len"]
        self.tok_emb = nn.Embedding(vocab_size, cfg["d_model"])
        self.drop    = nn.Dropout(cfg["dropout"])
        self.blocks  = nn.ModuleList([BiScaleLegalBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.norm    = RMSNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], vocab_size, bias=False)


        self.lm_head.weight = self.tok_emb.weight


        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
# ======================================= end of copied code =======================================


_ORIGINAL_ATTN_FORWARD = BiScaleGQA.forward


def _sdpa_attn_forward(self, x):
    B, T, C = x.shape
    dh = self.d_head
    q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    cos, sin = self.rope(T)
    q = apply_rotary(q, cos.to(q.dtype), sin.to(q.dtype))
    k = apply_rotary(k, cos.to(k.dtype), sin.to(k.dtype))
    k = k.repeat_interleave(self.n_groups, dim=1)
    v = v.repeat_interleave(self.n_groups, dim=1)
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=self.mask[:, :, :T, :T].bool(),
                                       dropout_p=0.0, scale=dh ** -0.5)
    return self.o_proj(y.transpose(1, 2).contiguous().view(B, T, -1))


def use_sdpa(flag=True):
    BiScaleGQA.forward = _sdpa_attn_forward if flag else _ORIGINAL_ATTN_FORWARD


class BiScaleForHarness(BiScaleLegalLLaMA):
    def forward(self, input_ids, prefix_lens=None):
        return super().forward(input_ids)[0]


def build_model(model_cfg):
    use_sdpa(model_cfg.get("use_sdpa", True))
    return BiScaleForHarness(model_cfg, model_cfg["vocab_size"])

Overwriting p1_biscale.py


In [22]:
# 3 — data check on the REAL tokenizer: formats, end convention, sizes, label counts, alignment ceiling
!python test_ner_data.py | tail -2
import ner_data as nd, importlib; importlib.reload(nd)
from tokenizers import ByteLevelBPETokenizer
_tok = ByteLevelBPETokenizer(vocab="tokenizer/vocab.json", merges="tokenizer/merges.txt")
def encode(t):
    e = _tok.encode(t); return e.ids, e.offsets
LABELS = nd.read_labels(P2["labels.txt"])
print("labels:", LABELS, "| matches the document's 12:", sorted(LABELS) == sorted(nd.LABELS_DEFAULT))
raw = json.load(open(P2["fold1-gold.json"]))
first = next(iter(raw.items())) if isinstance(raw, dict) else raw[0]
print("gold layout example:", str(first)[:300])
TX, GD = {}, {}
for k in ("fold1", "fold2"):
    t, g = nd.load_split(P2[f"{k}-text.json"], P2[f"{k}-gold.json"])
    conv = nd.detect_end_convention(t, g)
    print(f"\n{k}: end convention {conv}")
    TX.update(t); GD.update(g)
INCL = nd.detect_end_convention(TX, GD)["inclusive"]
SP = nd.to_internal(GD, INCL)
st = nd.dataset_stats(TX, SP, encode, LABELS)
print("\ntrain:", json.dumps(st, indent=1))
TT = nd.load_split(P2["fold3-text.json"])[0]
print("test:", {k: v for k, v in nd.dataset_stats(TT, None, encode, LABELS).items() if not k.startswith("per")})
for scheme in ("IO", "BIO", "BIOES"):
    sc, sk, mid = nd.oracle_check(TX, SP, encode, LABELS, scheme)
    print(f"\nORACLE {scheme}: strict macro-F1 {sc['macro_f1']:.4f} | overlapping skipped {sk} | "
          f"entities with a boundary inside a token: {dict(mid)}")
print("\nper-label oracle (BIO):\n" + nd.format_scores(nd.oracle_check(TX, SP, encode, LABELS, "BIO")[0]))


ALL TESTS PASSED
labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO'] | matches the document's 12: True
gold layout example: ('115651329', [[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana'], [359, 369, 'COURT', 'High Court'], [575, 585, 'DATE', '18.11.1994'], [637, 644, 'WIT', 'Jagdish'], [695, 704, 'WIT', 'Mange Ram'], [792, 800, 'APP', 'Jage 

fold1: end convention {'inclusive': False, 'exclusive_matches': 4120, 'inclusive_matches': 0, 'neither': 0, 'with_text': 4120, 'examples': []}

fold2: end convention {'inclusive': False, 'exclusive_matches': 4273, 'inclusive_matches': 0, 'neither': 0, 'with_text': 4273, 'examples': []}

train: {
 "docs": 70,
 "tokens": 465720,
 "doc_tokens_p50": 5577,
 "doc_tokens_p90": 11317,
 "doc_tokens_max": 23915,
 "entities": 8393,
 "per_label": {
  "DATE": 1575,
  "STAT": 1260,
  "PREC": 1183,
  "COURT": 1013,
  

In [23]:
# 4 — Phase-1 models -> phase1_registry.json (+ p1_<name>.py copies of each model.py)
REG = {}
V = _tok.get_vocab_size()
def near(ckpt, fn):
    root = os.path.dirname(os.path.dirname(ckpt)) if os.path.basename(os.path.dirname(ckpt)) == "run" else os.path.dirname(ckpt)
    hits = sorted(glob.glob(f"{root}/**/{fn}", recursive=True), key=len)
    return hits[0] if hits else None
for name in ["prefixlm", "encdec", "arch4"]:
    ck = find(f"{name}_best.pt")
    if not ck:
        print(f"{name}: NOT FOUND (attach its run output)"); continue
    mp, rc = near(ck[0], "model.py"), near(ck[0], "run_config.json")
    shutil.copy(mp, f"p1_{name}.py")
    cfgj = json.load(open(rc))
    mc = dict(cfgj["model"]); mc.setdefault("vocab_size", V); mc.setdefault("max_seq_len", int(cfgj["train"]["ctx_len"]))
    REG[name] = dict(module=f"p1_{name}", model_cfg=mc, ckpt=ck[0])
bs = find("stage2_done.pt") + find("best_model.pt")
if bs:
    REG["biscale"] = dict(module="p1_biscale", ckpt=bs[0],
                          model_cfg=dict(n_layers=16, n_heads=12, n_kv_heads=4, d_model=768, d_ff=2048, ctx_len=1024,
                                         local_window=256, dropout=0.1, rope_theta=10000.0, vocab_size=V))
else:
    print("biscale: NOT FOUND (attach the extended Bi-Scale run output with stage2_done.pt)")
json.dump(REG, open("phase1_registry.json", "w"), indent=1)
from phase1_loader import load_phase1
for name in REG:
    m, info = load_phase1(name)
    print(f"{name:9s} {sum(p.numel() for p in m.parameters())/1e6:7.2f}M params | {info['ckpt']}")
    del m

prefixlm   109.63M params | /kaggle/input/datasets/karanchauhan435257/phase1-models/prefixlm-final/run/prefixlm_best.pt
encdec     114.37M params | /kaggle/input/datasets/karanchauhan435257/phase1-models/encdec-final/run/encdec_best.pt
arch4      109.60M params | /kaggle/input/datasets/karanchauhan435257/phase1-models/arch4-final/run/arch4_best.pt
biscale    144.16M params | /kaggle/input/datasets/karanchauhan435257/phase1-models/biscale-final/checkpoints/stage2_done.pt


In [24]:
# 5 — experiment config helper
TODO = "TODO"
def make_cfg(run_name, model, train, data=None, kd=None, dump_teacher=False, module="ner_model"):
    d = dict(tokenizer_dir="tokenizer", labels=P2["labels.txt"],
             train_files=[[P2["fold1-text.json"], P2["fold1-gold.json"]], [P2["fold2-text.json"], P2["fold2-gold.json"]]],
             test_file=P2["fold3-text.json"],
             scheme=TODO, decode="lenient", window=TODO, stride=TODO,
             val_frac=0.15, split_seed=0)
    d.update(data or {})
    cfg = dict(run_name=run_name, out_dir=f"/kaggle/working/p2runs/{run_name}", model_module=module,
               data=d, model=model, train=train, kd=kd, dump_teacher_logits=dump_teacher)
    todo = [k for k, v in {**d, **train, **(kd or {}), **model}.items() if v == TODO]
    assert not todo, f"{run_name}: fill {todo}"
    os.makedirs(cfg["out_dir"], exist_ok=True)
    p = f"{cfg['out_dir']}/config.json"; json.dump(cfg, open(p, "w"), indent=1)
    return p

def run_parallel(cfg_paths, timeout_min=180):
    procs, queue = [], list(cfg_paths)
    while queue or procs:
        while queue and len(procs) < max(1, NGPU):
            cp = queue.pop(0); gpu = [g for g in range(max(1, NGPU)) if g not in [x[2] for x in procs]][0]
            lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            procs.append((subprocess.Popen([sys.executable, "ner_train.py", cp], stdout=lf, stderr=subprocess.STDOUT,
                                           env=env), cp, gpu, lf, time.time()))
            print(f"started {cp} on GPU {gpu}")
        time.sleep(20)
        for pr in list(procs):
            p, cp, gpu, lf, t0 = pr
            if p.poll() is not None or time.time() - t0 > timeout_min * 60:
                if p.poll() is None: p.kill()
                lf.close(); procs.remove(pr)
                tail = open(os.path.join(os.path.dirname(cp), "train.log")).read().splitlines()[-25:]
                print(f"\n==== {cp} exit {p.returncode} ({(time.time()-t0)/60:.1f} min)\n" + "\n".join(tail))

In [25]:
# 6 — smoke test: dummy model, 1 epoch, whole pipeline (train -> strict val F1 -> teacher dump -> KD -> test zip)
SMOKE_T = dict(epochs=1, lr=1e-3, batch_size=8, grad_accum=1, warmup_frac=0.0, weight_decay=0.0, max_minutes=3)
SMOKE_D = dict(scheme="BIO", window=256, stride=192)
a = make_cfg("smoke_teacher", dict(vocab_size=V), SMOKE_T, SMOKE_D, dump_teacher=True, module="ner_dummy")
run_parallel([a])
b = make_cfg("smoke_student", dict(vocab_size=V), SMOKE_T, SMOKE_D, module="ner_dummy",
             kd=dict(teacher_logits="/kaggle/working/p2runs/smoke_teacher/smoke_teacher_teacher_logits.npz",
                     temperature=2.0, alpha=0.5))
run_parallel([b])
ok = all(os.path.exists(f"/kaggle/working/p2runs/{r}/{r}_submission.zip") for r in ("smoke_teacher", "smoke_student"))
assert ok, "SMOKE TEST FAILED — read the logs above"
print("SMOKE TEST PASSED (pipeline only: the dummy model's scores mean nothing)")

started /kaggle/working/p2runs/smoke_teacher/config.json on GPU 0

==== /kaggle/working/p2runs/smoke_teacher/config.json exit 0 (0.3 min)
labels 12 | scheme BIO -> 25 tags | gold end is exclusive (8393 exclusive / 0 inclusive / 0 neither)
train docs 63 | val docs 7 | test docs 35
parameters: total 2.05M | trainable 2.050M (100.00%)
train windows 2199 (window 256, stride 192) | overlapping gold entities not representable: 0
epoch 1 | train_loss 2.1391 | minutes 0.0217 | val_macro_f1 0.0090 | val_macro_p 0.0049 | val_macro_r 0.0585
  best so far (epoch 1)
BEST checkpoint on validation:
strict macro-F1 0.0090 (over labels present: 0.0090) | macro-P 0.0049 | macro-R 0.0585
  APP        F1 0.000  P 0.000  R 0.000  (tp 0, fp 809, fn 36, support 36)
  RESP       F1 0.000  P 0.000  R 0.000  (tp 0, fp 268, fn 31, support 31)
  A.COUNSEL  F1 0.000  P 0.000  R 0.000  (tp 0, fp 507, fn 16, support 16)
  R.COUNSEL  F1 0.000  P 0.000  R 0.000  (tp 0, fp 621, fn 15, support 15)
  JUDGE      F1 0.000 

In [26]:
%%writefile ner_model.py
import math
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from phase1_loader import load_phase1

BLOCKS = {"prefixlm": "blocks", "encdec": "encoder", "arch4": "blocks", "biscale": "blocks"}
FINAL_NORM = {"prefixlm": "ln_f", "encdec": "enc_ln", "arch4": "ln_f", "biscale": "norm"}
DEFAULT_TARGETS = {
    "prefixlm": ["attn.qkv", "attn.proj", "mlp.fc1", "mlp.fc2"],
    "encdec": ["attn.qkv", "attn.proj", "mlp.fc1", "mlp.fc2"],
    "arch4": ["msr.proj", "msr.out", "mlp.fc1", "mlp.fc2"],
    "biscale": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "ffn.gate", "ffn.up", "ffn.down"],
}
ADAPTER_SITES = {
    "prefixlm": ("attn", "mlp"), "encdec": ("attn", "mlp"),
    "arch4": ("msr", "mlp"), "biscale": ("attn", "ffn"),
}


def _first_attr(obj, names):
    for n in names:
        if getattr(obj, n, None) is not None:
            return getattr(obj, n)
    raise AttributeError(f"{type(obj).__name__} has none of {names}")


def _out_splits(backbone_name, path, lin, d_model):
    if path.endswith("attn.qkv"):
        return [lin.out_features // 3] * 3
    if backbone_name == "arch4" and path.endswith("msr.proj"):
        vd = (lin.out_features - 2 * d_model) // 2
        return [d_model, d_model, vd, vd]
    return [lin.out_features]


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, splits, variant):
        super().__init__()
        self.base, self.splits, self.variant = base, list(splits), variant
        self.scale = alpha / r
        self.A = nn.ParameterList()
        self.B = nn.ParameterList()
        for d in self.splits:
            if variant == "plus":
                A = torch.empty(r, d)
                nn.init.kaiming_uniform_(A, a=math.sqrt(5))
                B = torch.zeros(d, r)
            elif variant == "pi":
                A = torch.zeros(r, d)
                A[0, 0] = 1.0
                B = torch.randn(d, r) / math.sqrt(r)
                B[:, 0] = 1.0
            else:
                raise ValueError(f"lorma_variant must be 'plus' or 'pi', got {variant!r}")
            self.A.append(nn.Parameter(A))
            self.B.append(nn.Parameter(B))

    def forward(self, x):
        y = F.linear(x, self.base.weight)
        outs = []
        for i, part in enumerate(y.split(self.splits, dim=-1)):
            A, B = self.A[i], self.B[i]
            if self.variant == "plus":
                outs.append(part + F.linear(F.linear(part, A.to(part.dtype)), B.to(part.dtype)) * self.scale)
            else:
                M = B @ A
                d = M.shape[0]
                idx = torch.arange(d, device=M.device)
                R = torch.gather(M, 1, (idx[None, :] - idx[:, None]) % d)
                outs.append(F.linear(part, R.to(part.dtype)))
        y = torch.cat(outs, dim=-1) if len(outs) > 1 else outs[0]
        if self.base.bias is not None:
            y = y + self.base.bias.to(y.dtype)
        return y


class Adapter(nn.Module):
    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.normal_(self.down.weight, 0.0, 0.02)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class WithAdapter(nn.Module):
    def __init__(self, inner, d, m):
        super().__init__()
        self.inner = inner
        self.adapter = Adapter(d, m)

    def forward(self, *args, **kwargs):
        out = self.inner(*args, **kwargs)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["inner"], name)


class AttnContext:
    def __init__(self):
        self.key_mask = None


class EncoderAttention(nn.Module):
    def __init__(self, orig, ctx, rope_fn):
        super().__init__()
        self.orig, self.ctx, self.rope_fn = orig, ctx, rope_fn

    def forward(self, x, cos, sin, causal=False, cache=None, li=None, start=0):
        o = self.orig
        B, T, _ = x.shape
        q, k, v = o.qkv(x).view(B, T, 3, o.h, o.hd).permute(2, 0, 3, 1, 4)
        q, k = self.rope_fn(q, cos, sin), self.rope_fn(k, cos, sin)
        km = self.ctx.key_mask
        mask = None if km is None else km[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           dropout_p=o.p if self.training else 0.0)
        return o.proj(y.transpose(1, 2).reshape(B, T, o.d))


class NERHead(nn.Module):
    def __init__(self, d, num_labels, kind, hidden, dropout, lookahead):
        super().__init__()
        self.kind = kind
        self.drop = nn.Dropout(dropout)
        self.context = None
        if lookahead:
            self.context = nn.Conv1d(d, d, kernel_size=2 * lookahead + 1, padding=lookahead)
            nn.init.zeros_(self.context.weight)
            nn.init.zeros_(self.context.bias)
        if kind == "linear":
            self.body, out_d = None, d
        elif kind == "mlp":
            self.body, out_d = nn.Sequential(nn.Linear(d, hidden), nn.GELU(), nn.Dropout(dropout)), hidden
        else:
            raise ValueError(f"head must be linear | mlp, got {kind!r}")
        self.out = nn.Linear(out_d, num_labels)
        nn.init.normal_(self.out.weight, 0.0, 0.02)
        nn.init.zeros_(self.out.bias)

    def forward(self, h, attention_mask):
        h = h * attention_mask[..., None].to(h.dtype)
        if self.context is not None:
            h = h + self.context(h.transpose(1, 2)).transpose(1, 2)
        h = self.drop(h)
        if self.kind == "mlp":
            h = self.body(h)
        return self.out(h)


class NERModel(nn.Module):
    def __init__(self, cfg, num_labels):
        super().__init__()
        self.cfg = cfg
        self.name = cfg.get("phase1", "prefixlm")
        if self.name not in BLOCKS:
            raise ValueError(f"phase1 must be one of {list(BLOCKS)}, got {self.name!r}")
        self.method = cfg.get("method", "full")
        backbone, self.info = load_phase1(self.name)
        self.backbone = backbone
        self.p1 = sys.modules[type(backbone).__module__]
        self.ctx = AttnContext()
        self.grad_ckpt = bool(cfg.get("grad_ckpt", False))

        self.emb_name = next(n for n in ("tok_emb", "embed_tokens", "embed", "wte")
                             if isinstance(getattr(backbone, n, None), nn.Embedding))
        self.d = self.emb.embedding_dim
        self._strip_lm_head()
        self._make_student()
        if cfg.get("backbone_dropout") is not None:
            for mod in backbone.modules():
                if isinstance(mod, nn.Dropout):
                    mod.p = float(cfg["backbone_dropout"])
        if self.name == "encdec":
            self._wrap_encoder_attention()

        self.n_prompt = 0
        self.prompt = None
        self._apply_method()

        self.head = NERHead(self.d, num_labels, cfg.get("head", "linear"), int(cfg.get("head_hidden", 512)),
                            float(cfg.get("head_dropout", 0.1)), int(cfg.get("lookahead", 0)))

    @property
    def emb(self):
        return getattr(self.backbone, self.emb_name)

    def _blocks(self):
        return getattr(self.backbone, BLOCKS[self.name])

    def _stacks(self):
        return [self._blocks()]

    def _strip_lm_head(self):
        bb = self.backbone
        V = self.emb.num_embeddings
        for n, mod in list(bb.named_children()):
            if isinstance(mod, nn.Linear) and mod.out_features == V:
                setattr(bb, n, None)
        if self.name == "encdec":
            bb.decoder = None
            bb.dec_ln = None

    def _make_student(self):
        k = self.cfg.get("keep_layers")
        if not k:
            return
        blocks = self._blocks()
        n = len(blocks)
        k = int(k)
        if self.cfg.get("keep_mode", "first") == "spread":
            idx = sorted({round(i * (n - 1) / max(1, k - 1)) for i in range(k)})
        else:
            idx = list(range(k))
        setattr(self.backbone, BLOCKS[self.name], nn.ModuleList([blocks[i] for i in idx]))
        self.kept_layers = idx

    def _wrap_encoder_attention(self):
        for blk in self.backbone.encoder:
            blk.attn = EncoderAttention(blk.attn, self.ctx, self.p1.apply_rope)

    def _apply_method(self):
        cfg, m = self.cfg, self.method
        bb = self.backbone
        if m == "full":
            for p in bb.parameters():
                p.requires_grad = True
            return
        for p in bb.parameters():
            p.requires_grad = False
        if m == "partial":
            n_top = int(cfg.get("unfreeze_layers", 4))
            for stack in self._stacks():
                for blk in list(stack)[-n_top:]:
                    for p in blk.parameters():
                        p.requires_grad = True
            for p in getattr(bb, FINAL_NORM[self.name]).parameters():
                p.requires_grad = True
            if cfg.get("unfreeze_embeddings", False):
                self.emb.weight.requires_grad = True
        elif m == "bitfit":
            for n, p in bb.named_parameters():
                if n.endswith(".bias"):
                    p.requires_grad = True
        elif m in ("lora", "lorma"):
            self._wrap_linears(m)
        elif m == "adapter":
            self._add_adapters()
        elif m == "prompt":
            self._add_prompt()
        else:
            raise ValueError(f"unknown method {m!r}")
        if cfg.get("train_norms", False):
            for mod in bb.modules():
                if isinstance(mod, (nn.LayerNorm, nn.GroupNorm)) or type(mod).__name__.lower().endswith("rmsnorm"):
                    for p in mod.parameters(recurse=False):
                        p.requires_grad = True

    def _wrap_linears(self, kind):
        cfg = self.cfg
        targets = cfg.get("targets") or DEFAULT_TARGETS[self.name]
        r = int(cfg.get("rank", 8))
        alpha = float(cfg.get("alpha", 2 * r))
        stack_names = [BLOCKS[self.name]]
        replaced = []
        for sname in stack_names:
            for bi, blk in enumerate(getattr(self.backbone, sname)):
                for path, mod in list(blk.named_modules()):
                    clean = path.replace("orig.", "").replace("inner.", "")
                    if not isinstance(mod, nn.Linear) or not any(clean == t or clean.endswith("." + t)
                                                                 for t in targets):
                        continue
                    parent_path, _, leaf = path.rpartition(".")
                    parent = blk.get_submodule(parent_path) if parent_path else blk
                    if kind == "lora":
                        new = LoRALinear(mod, r, alpha, float(cfg.get("lora_dropout", 0.05)))
                    else:
                        splits = _out_splits(self.name, clean, mod, self.d) if cfg.get("lorma_split", True) \
                            else [mod.out_features]
                        new = LoRMALinear(mod, r, alpha, splits, cfg.get("lorma_variant", "plus"))
                    setattr(parent, leaf, new)
                    replaced.append(f"{sname}.{bi}.{path}")
        if not replaced:
            raise ValueError(f"no linear matched targets {targets}")
        self.adapted = replaced

    def _add_adapters(self):
        m = int(self.cfg.get("adapter_size", 64))
        style = self.cfg.get("adapter_style", "houlsby")
        attn_site, mlp_site = ADAPTER_SITES[self.name]
        sites = [attn_site, mlp_site] if style == "houlsby" else [mlp_site]
        for stack in self._stacks():
            for blk in stack:
                for s in sites:
                    setattr(blk, s, WithAdapter(getattr(blk, s), self.d, m))

    def _add_prompt(self):
        n = int(self.cfg.get("n_prompt", 20))
        g = torch.Generator().manual_seed(int(self.cfg.get("init_seed", 0)))
        ids = torch.randint(1000, self.emb.num_embeddings, (n,), generator=g)
        self.prompt = nn.Parameter(self.emb.weight.detach()[ids].clone().float())
        self.n_prompt = n

    def hidden(self, input_ids, attention_mask):
        bb = self.backbone
        B, W = input_ids.shape
        x = self.emb(input_ids)
        am = attention_mask.bool()
        if self.prompt is not None:
            x = torch.cat([self.prompt.to(x.dtype)[None].expand(B, -1, -1), x], dim=1)
            am = torch.cat([am.new_ones(B, self.n_prompt), am], dim=1)
        T = x.shape[1]
        if T > self.backbone_max_len():
            raise ValueError(f"window + prompt = {T} exceeds the Phase-1 context {self.backbone_max_len()}")
        h = getattr(self, f"_hidden_{self.name}")(x, am, T)
        return h[:, T - W:]

    def backbone_max_len(self):
        bb = self.backbone
        for n in ("max_len", "max_seq_len"):
            if isinstance(getattr(bb, n, None), int):
                return getattr(bb, n)
        c = getattr(bb, "c", None) or getattr(bb, "cfg", None) or {}
        if isinstance(c, dict) and c.get("max_seq_len"):
            return int(c["max_seq_len"])
        return 1024

    def _run(self, blk, *args):
        if self.grad_ckpt and self.training:
            return checkpoint(blk, *args, use_reentrant=False)
        return blk(*args)

    def _hidden_prefixlm(self, x, am, T):
        bb = self.backbone
        if bb.pos_emb is not None:
            x = x + bb.pos_emb(torch.arange(T, device=x.device))
        x = bb.drop(x)
        cos = bb.rope_cos[:T] if bb.rope_cos is not None else None
        sin = bb.rope_sin[:T] if bb.rope_sin is not None else None
        mask = self.p1.prefix_lm_mask(T, am.sum(1), x.device)
        for blk in bb.blocks:
            x = self._run(blk, x, cos, sin, mask)
        return bb.ln_f(x)

    def _hidden_encdec(self, x, am, T):
        bb = self.backbone
        self.ctx.key_mask = None if bool(am.all()) else am
        try:
            x = bb.drop(x)
            cos, sin = bb.rope_cos[:T], bb.rope_sin[:T]
            for blk in bb.encoder:
                x = self._run(blk, x, cos, sin)
            return bb.enc_ln(x)
        finally:
            self.ctx.key_mask = None

    def _hidden_arch4(self, x, am, T):
        bb = self.backbone
        t = bb.tables(x.device)
        x = bb.drop(x)
        cos, sin, mask = t["cos"][:T], t["sin"][:T], t["mask"][:, :T, :T]
        for blk in bb.blocks:
            x = self._run(blk, x, cos, sin, mask)
        return bb.ln_f(x)

    def _hidden_biscale(self, x, am, T):
        bb = self.backbone
        drop = getattr(bb, "drop", None)
        if drop is not None:
            x = drop(x)
        for blk in bb.blocks:
            x = self._run(blk, x)
        return _first_attr(bb, ["norm", "ln_f", "final_norm"])(x)

    def forward(self, input_ids, attention_mask=None):
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids, dtype=torch.bool)
        h = self.hidden(input_ids, attention_mask)
        return self.head(h, attention_mask.bool())

    def param_groups(self, base_lr):
        head_mult = float(self.cfg.get("head_lr_mult", 1.0))
        new_mult = float(self.cfg.get("peft_lr_mult", 1.0))
        bb_mult = float(self.cfg.get("backbone_lr_mult", 1.0))
        wd = self.cfg.get("weight_decay")
        groups = {"head": [], "head_nd": [], "new": [], "new_nd": [], "bb": [], "bb_nd": []}
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            nd = p.dim() < 2 or n == "prompt"
            if n.startswith("head."):
                key = "head"
            elif n == "prompt" or n.endswith((".A", ".B")) or ".A." in n or ".B." in n or ".adapter." in n:
                key = "new"
            else:
                key = "bb"
            groups[key + ("_nd" if nd else "")].append(p)
        mult = {"head": head_mult, "new": new_mult, "bb": bb_mult}
        out = []
        for k, ps in groups.items():
            if not ps:
                continue
            g = {"params": ps, "lr": base_lr * mult[k.split("_")[0]]}
            if k.endswith("_nd"):
                g["weight_decay"] = 0.0
            elif wd is not None:
                g["weight_decay"] = float(wd)
            out.append(g)
        return out


def build_ner_model(model_cfg: dict, num_labels: int) -> nn.Module:
    return NERModel(dict(model_cfg), int(num_labels))


@torch.no_grad()
def check_init_equivalence(model_cfg: dict, num_labels: int, window: int = 256, batch: int = 2, seed: int = 0):
    adapted = NERModel(dict(model_cfg), num_labels).eval()
    plain = NERModel(dict(model_cfg, method="full", keep_layers=model_cfg.get("keep_layers")), num_labels).eval()
    g = torch.Generator().manual_seed(seed)
    ids = torch.randint(1000, adapted.emb.num_embeddings, (batch, window), generator=g)
    dev = adapted.emb.weight.device
    plain.to(dev)
    am = torch.ones_like(ids, dtype=torch.bool).to(dev)
    diff = (adapted.hidden(ids.to(dev), am).float() - plain.hidden(ids.to(dev), am).float()).abs().max().item()
    n_all = sum(p.numel() for p in adapted.parameters())
    n_tr = sum(p.numel() for p in adapted.parameters() if p.requires_grad)
    print(f"{model_cfg.get('phase1')}/{model_cfg.get('method')}: max |hidden(adapted) - hidden(pretrained)| = "
          f"{diff:.2e} | trainable {n_tr/1e6:.3f}M of {n_all/1e6:.2f}M")
    return diff


Writing ner_model.py


In [27]:
   import importlib, ner_model; importlib.reload(ner_model)
   C = len(nd.label_names(LABELS, "BIO"))
   for mc in [dict(phase1="prefixlm", method="lorma"), dict(phase1="encdec", method="lora"),
              dict(phase1="arch4", method="adapter"), dict(phase1="prefixlm", method="bitfit"),
              dict(phase1="biscale", method="partial")]:
       ner_model.check_init_equivalence(mc, C)

prefixlm/lorma: max |hidden(adapted) - hidden(pretrained)| = 1.02e-05 | trainable 1.346M of 110.98M
encdec/lora: max |hidden(adapted) - hidden(pretrained)| = 0.00e+00 | trainable 0.609M of 67.71M
arch4/adapter: max |hidden(adapted) - hidden(pretrained)| = 0.00e+00 | trainable 1.804M of 111.41M
prefixlm/bitfit: max |hidden(adapted) - hidden(pretrained)| = 0.00e+00 | trainable 0.121M of 109.65M
biscale/partial: max |hidden(adapted) - hidden(pretrained)| = 0.00e+00 | trainable 29.917M of 144.18M


In [28]:
# 8 — model check: shapes, trainable parameters, a training step, memory (run before the experiments)
import importlib, ner_model; importlib.reload(ner_model)
CHECK_MODEL = dict(TODO="TODO")
C = len(nd.label_names(LABELS, "BIO"))
m = ner_model.build_ner_model(dict(CHECK_MODEL), C).cuda().train()
x = torch.randint(5, V, (4, 512), device="cuda"); msk = torch.ones_like(x, dtype=torch.bool)
torch.cuda.reset_peak_memory_stats()
with torch.autocast("cuda", dtype=torch.float16):
    out = m(x, msk)
assert tuple(out.shape) == (4, 512, C), f"logits must be [B, W, num_labels], got {tuple(out.shape)}"
out.float().mean().backward()
tr = [n for n, p in m.named_parameters() if p.requires_grad]
nog = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
print(f"total {sum(p.numel() for p in m.parameters())/1e6:.2f}M | trainable "
      f"{sum(p.numel() for p in m.parameters() if p.requires_grad)/1e6:.3f}M in {len(tr)} tensors | "
      f"peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB at batch 4 x 512")
print("trainable but got no gradient (unused?):", nog[:10])
del m, out; torch.cuda.empty_cache()

total 109.65M | trainable 109.651M in 149 tensors | peak 1.5 GB at batch 4 x 512
trainable but got no gradient (unused?): []


In [29]:
# 9 — EXPERIMENTS: six methods across all four Phase-1 models; two runs at a time (one per GPU).


TODO = "TODO"

DATA = dict(scheme="BIO", window=512, stride=384)
TRAIN = dict(epochs=10, lr=5e-5, batch_size=8, grad_accum=1, warmup_frac=0.1,
             weight_decay=0.01, schedule="linear", class_weights=None, max_minutes=60)

def T(**over):
    return dict(TRAIN, **over)

R = "/kaggle/working/p2runs"
PLAN = {

    "prefixlm_lorma":  dict(model=dict(phase1="prefixlm", method="lorma", lorma_variant="plus", rank=16, alpha=32,
                                       head_lr_mult=2.0),
                            train=T(lr=5e-4, epochs=15)),
    "encdec_lora":     dict(model=dict(phase1="encdec", method="lora", rank=16, alpha=32, lora_dropout=0.05,
                                       head_lr_mult=2.0),
                            train=T(lr=5e-4, epochs=15)),

    "biscale_full":    dict(model=dict(phase1="biscale", method="full", lookahead=4, head_lr_mult=20.0),
                            train=T(), dump_teacher=True),
    "arch4_adapter":   dict(model=dict(phase1="arch4", method="adapter", adapter_size=64, adapter_style="houlsby",
                                       lookahead=4, head_lr_mult=1.0),
                            train=T(lr=1e-3, epochs=15)),

    "prefixlm_bitfit": dict(model=dict(phase1="prefixlm", method="bitfit", head_lr_mult=1.0),
                            train=T(lr=1e-3, epochs=15)),
    "kd_student":      dict(model=dict(phase1="biscale", method="full", keep_layers=8, keep_mode="spread",
                                       lookahead=4, head_lr_mult=20.0),
                            train=T(), kd=dict(teacher_logits=f"{R}/biscale_full/biscale_full_teacher_logits.npz",
                                               temperature=2.0, alpha=0.5)),
}
ROUNDS = [["prefixlm_lorma", "encdec_lora"], ["biscale_full", "arch4_adapter"], ["prefixlm_bitfit", "kd_student"]]

def cfg_for(name):
    p = PLAN[name]
    return make_cfg(name, model=p["model"], train=p["train"], data=DATA,
                    kd=p.get("kd"), dump_teacher=p.get("dump_teacher", False))

def run_round(i):
    names = ROUNDS[i]
    for n in names:
        kd = PLAN[n].get("kd")
        assert not kd or os.path.exists(kd["teacher_logits"]), f"{n}: teacher logits missing -> run the teacher first"
    run_parallel([cfg_for(n) for n in names])
    for n in names:
        z = f"{R}/{n}/{n}_submission.zip"
        res = f"{R}/{n}/{n}_result.json"
        if os.path.exists(z) and os.path.exists(res):
            r = json.load(open(res))
            print(f"OK  {n:16s} val macro-F1 {r.get('val', {}).get('macro_f1', float('nan')):.4f} | "
                  f"trainable {r['params_trainable']/1e6:.2f}M of {r['params_total']/1e6:.1f}M | "
                  f"{r['train_minutes']:.1f} min -> upload {z}")
        else:
            print(f"FAILED {n}: read {R}/{n}/train.log")


assert len({PLAN[n]["model"]["method"] for n in PLAN}) >= 5
assert {PLAN[n]["model"]["phase1"] for n in PLAN} == {"prefixlm", "encdec", "arch4", "biscale"}
_todo = lambda o: o == TODO or (isinstance(o, dict) and any(_todo(v) for v in o.values()))
assert not _todo(dict(DATA=DATA, TRAIN=TRAIN, PLAN=PLAN)), "a TODO is still left"

run_round(0)

started /kaggle/working/p2runs/prefixlm_lorma/config.json on GPU 0
started /kaggle/working/p2runs/encdec_lora/config.json on GPU 1

==== /kaggle/working/p2runs/encdec_lora/config.json exit 0 (4.3 min)
epoch 11 | train_loss 0.2029 | minutes 2.8474 | val_macro_f1 0.1087 | val_macro_p 0.0980 | val_macro_r 0.1291
  best so far (epoch 11)
epoch 12 | train_loss 0.1959 | minutes 3.1178 | val_macro_f1 0.1191 | val_macro_p 0.1011 | val_macro_r 0.1548
  best so far (epoch 12)
epoch 13 | train_loss 0.1903 | minutes 3.3865 | val_macro_f1 0.1236 | val_macro_p 0.1217 | val_macro_r 0.1578
  best so far (epoch 13)
epoch 14 | train_loss 0.1856 | minutes 3.6548 | val_macro_f1 0.1140 | val_macro_p 0.1020 | val_macro_r 0.1353
epoch 15 | train_loss 0.1814 | minutes 3.9235 | val_macro_f1 0.1197 | val_macro_p 0.1037 | val_macro_r 0.1484
BEST checkpoint on validation:
strict macro-F1 0.1236 (over labels present: 0.1236) | macro-P 0.1217 | macro-R 0.1578
  APP        F1 0.050  P 0.250  R 0.028  (tp 1, fp 3, fn

In [30]:
import zipfile, json
from IPython.display import FileLink, display
for r in ["prefixlm_lorma", "encdec_lora"]:
    z = f"p2runs/{r}/{r}_submission.zip"
    with zipfile.ZipFile(z) as f:
        names = f.namelist()
        pred = json.loads(f.read("prediction.json"))
    k = next(iter(pred))
    print(r, "| zip contains:", names, "| docs:", len(pred), "| example:", k, pred[k][:2])
    display(FileLink(z))

prefixlm_lorma | zip contains: ['prediction.json'] | docs: 35 | example: 10104667 [[107, 136, 'APP'], [157, 172, 'RESP']]


/kaggle/working/p2runs/prefixlm_lorma/prefixlm_lorma_submission.zip

encdec_lora | zip contains: ['prediction.json'] | docs: 35 | example: 10104667 [[90, 93, 'CASENO'], [97, 101, 'CASENO']]


/kaggle/working/p2runs/encdec_lora/encdec_lora_submission.zip

In [31]:
run_round(1)

started /kaggle/working/p2runs/biscale_full/config.json on GPU 0
started /kaggle/working/p2runs/arch4_adapter/config.json on GPU 1

==== /kaggle/working/p2runs/arch4_adapter/config.json exit 0 (7.7 min)
epoch 10 | train_loss 0.0008 | minutes 4.7357 | val_macro_f1 0.3504 | val_macro_p 0.3274 | val_macro_r 0.3885
epoch 11 | train_loss 0.0005 | minutes 5.2334 | val_macro_f1 0.3522 | val_macro_p 0.3323 | val_macro_r 0.3853
  best so far (epoch 11)
epoch 12 | train_loss 0.0005 | minutes 5.7282 | val_macro_f1 0.3642 | val_macro_p 0.3422 | val_macro_r 0.4002
  best so far (epoch 12)
epoch 13 | train_loss 0.0004 | minutes 6.2212 | val_macro_f1 0.3611 | val_macro_p 0.3410 | val_macro_r 0.3958
epoch 14 | train_loss 0.0004 | minutes 6.7127 | val_macro_f1 0.3631 | val_macro_p 0.3421 | val_macro_r 0.3988
epoch 15 | train_loss 0.0003 | minutes 7.2054 | val_macro_f1 0.3638 | val_macro_p 0.3442 | val_macro_r 0.3972
BEST checkpoint on validation:
strict macro-F1 0.3642 (over labels present: 0.3642) | m

In [32]:
# 9b — UPDATE (run after the current round finishes): Viterbi decoding option + re-score finished runs without retraining
open("ner_data.py", "w").write('"""\nner_data.py — Phase-2 L-NER data handling.\n\n  load_split(text_path, gold_path)   documents + gold entities, any of the common JSON layouts\n  detect_end_convention(...)         is the gold `end` exclusive (text[s:e]) or inclusive (text[s:e+1])?\n  Tokenized docs                     token ids + character offsets TRIMMED of whitespace\n                                     (ByteLevel BPE puts the leading space inside a token: " Court" -> its\n                                     raw offset starts at the space; untrimmed, every span start is off by 1)\n  token labels                       BIO or BIOES (or IO) from character spans\n  windows                            fixed-length token windows over long documents (with overlap)\n  decode_spans                       per-token label ids -> character spans\n  strict_scores                      exact (start, end, label) matching, per-label P/R/F1 and macro-F1\n  oracle_check                       gold -> token labels -> decode -> score: the ceiling of token-level\n                                     modelling with this tokenizer (should be ~1.0; what is lost is\n                                     entities whose boundary falls inside a token)\n\nInternally every span is (start, end_exclusive, label). Output is converted back to the gold convention.\n"""\nimport json\nimport zlib\nfrom collections import Counter, defaultdict\n\nLABELS_DEFAULT = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT",\n                  "STAT", "PREC", "DATE", "CASENO"]\n\n\n# ------------------------------------------------------------------ loading\ndef read_labels(path):\n    labs = []\n    for line in open(path, encoding="utf-8"):\n        line = line.strip()\n        if line:\n            labs += [x.strip() for x in line.replace(",", " ").split() if x.strip()]\n    return labs\n\n\ndef _as_text_map(obj):\n    if isinstance(obj, dict):\n        return {str(k): (v if isinstance(v, str) else v.get("text", "")) for k, v in obj.items()}\n    out = {}\n    for r in obj:\n        out[str(r.get("id", r.get("doc_id")))] = r["text"]\n    return out\n\n\ndef _parse_ent(e):\n    if isinstance(e, (list, tuple)):\n        s, en, lab = int(e[0]), int(e[1]), str(e[2])\n        txt = e[3] if len(e) > 3 else None\n    else:\n        s = int(e.get("start", e.get("start_char", e.get("begin"))))\n        en = int(e.get("end", e.get("end_char")))\n        lab = str(e.get("label", e.get("type", e.get("entity"))))\n        txt = e.get("text", e.get("entity_text"))\n    return s, en, lab, txt\n\n\ndef _as_gold_map(obj):\n    if isinstance(obj, dict):\n        items = obj.items()\n    else:\n        items = [(r.get("id", r.get("doc_id")), r.get("entities", r.get("spans", []))) for r in obj]\n    return {str(k): [_parse_ent(e) for e in v] for k, v in items}\n\n\ndef load_split(text_path, gold_path=None):\n    texts = _as_text_map(json.load(open(text_path, encoding="utf-8")))\n    gold = _as_gold_map(json.load(open(gold_path, encoding="utf-8"))) if gold_path else None\n    return texts, gold\n\n\ndef detect_end_convention(texts, gold):\n    """Counts how many gold entities match text[s:e] (exclusive end) vs text[s:e+1] (inclusive end)."""\n    exc = inc = neither = total = 0\n    examples = []\n    for d, ents in gold.items():\n        t = texts.get(d, "")\n        for s, e, lab, txt in ents:\n            if txt is None:\n                continue\n            total += 1\n            a, b = t[s:e] == txt, t[s:e + 1] == txt\n            exc += a; inc += (b and not a)\n            if not a and not b:\n                neither += 1\n                if len(examples) < 5:\n                    examples.append((d, s, e, lab, txt[:60], t[s:e + 1][:60]))\n    inclusive = inc > exc\n    return dict(inclusive=inclusive, exclusive_matches=exc, inclusive_matches=inc, neither=neither,\n                with_text=total, examples=examples)\n\n\ndef to_internal(gold, inclusive):\n    """-> {doc: [(s, e_exclusive, label)]}"""\n    return {d: [(s, e + 1 if inclusive else e, lab) for s, e, lab, _ in ents] for d, ents in gold.items()}\n\n\ndef from_internal(spans, inclusive):\n    return {d: [[s, e - 1 if inclusive else e, lab] for s, e, lab in v] for d, v in spans.items()}\n\n\ndef split_docs(doc_ids, val_frac, seed=0):\n    """Deterministic document-level split (same docs on every machine)."""\n    val = [d for d in doc_ids if (zlib.crc32(f"{seed}:{d}".encode()) % 10000) < val_frac * 10000]\n    vs = set(val)\n    return [d for d in doc_ids if d not in vs], val\n\n\n# ------------------------------------------------------------------ tokenization\ndef trim_offsets(text, offsets):\n    out = []\n    for s, e in offsets:\n        while s < e and text[s].isspace():\n            s += 1\n        while e > s and text[e - 1].isspace():\n            e -= 1\n        out.append((s, e))\n    return out\n\n\ndef tokenize_doc(text, encode):\n    """encode(text) -> (ids, offsets). Returns ids and whitespace-trimmed offsets; an all-space token has s == e."""\n    ids, offs = encode(text)\n    return list(ids), trim_offsets(text, offs)\n\n\n# ------------------------------------------------------------------ labels\ndef label_names(labels, scheme):\n    if scheme == "IO":\n        return ["O"] + [f"I-{l}" for l in labels]\n    if scheme == "BIO":\n        return ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]\n    if scheme == "BIOES":\n        return ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I", "E", "S")]\n    raise ValueError(scheme)\n\n\ndef token_labels(offsets, spans, labels, scheme):\n    """Character spans -> one tag id per token. A token belongs to an entity if it overlaps it.\n    Returns (tag_ids, n_overlapping_entities_skipped)."""\n    names = label_names(labels, scheme)\n    idx = {n: i for i, n in enumerate(names)}\n    tags = [0] * len(offsets)\n    owner = [None] * len(offsets)\n    skipped = 0\n    starts = [s for s, _ in offsets]\n    import bisect\n    for (S, E, lab) in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):\n        if lab not in labels:\n            continue\n        lo = max(0, bisect.bisect_left(starts, S) - 1)     # first of several tokens sharing a start\n        toks = []\n        j = lo\n        while j < len(offsets) and offsets[j][0] < E:\n            s, e = offsets[j]\n            if e > s and s < E and e > S:\n                toks.append(j)\n            j += 1\n        if not toks:\n            continue\n        if any(owner[t] is not None for t in toks):\n            skipped += 1                     # overlapping/nested entity: BIO cannot hold both\n            continue\n        # whitespace-only tokens strictly inside the entity are part of it too\n        full = list(range(toks[0], toks[-1] + 1))\n        for k, t in enumerate(full):\n            owner[t] = (S, E, lab)\n            if scheme == "IO":\n                name = f"I-{lab}"\n            elif scheme == "BIO":\n                name = f"{\'B\' if k == 0 else \'I\'}-{lab}"\n            else:\n                if len(full) == 1:\n                    name = f"S-{lab}"\n                else:\n                    name = f"{\'B\' if k == 0 else (\'E\' if k == len(full) - 1 else \'I\')}-{lab}"\n            tags[t] = idx[name]\n    return tags, skipped\n\n\ndef decode_spans(tag_ids, offsets, labels, scheme, mode="lenient"):\n    """Tag ids -> [(start, end_exclusive, label)] using the tokens\' trimmed character offsets.\n    mode \'lenient\': an I-/E- tag after O or a different label opens a new entity;\n    mode \'strict\' : only B-/S- open an entity (stray I-/E- are dropped)."""\n    names = label_names(labels, scheme)\n    if scheme == "IO":\n        mode = "lenient"                         # IO has no B-/S- tags: only the lenient rule applies\n    out = []\n    cur = None                                   # [label, start, end]\n\n    def close():\n        nonlocal cur\n        if cur is not None and cur[2] > cur[1]:\n            out.append((cur[1], cur[2], cur[0]))\n        cur = None\n\n    prev = None\n    for t, tid in enumerate(tag_ids):\n        s, e = offsets[t]\n        name = names[tid]\n        # ByteLevel BPE can split ONE character (e.g. an accented letter or a symbol) into several tokens\n        # with identical offsets: the later pieces never start or end a span on their own\n        same_char = prev is not None and (s, e) == prev and e > s\n        prev = (s, e)\n        if same_char and cur is not None:        # continuation of a character already in the open span\n            continue\n        if name == "O":\n            close(); continue\n        p, lab = name.split("-", 1)\n        if e <= s:                               # whitespace-only token: never opens, never ends a span\n            if cur is not None and cur[0] != lab:\n                close()\n            continue\n        if p in ("B", "S") or cur is None or cur[0] != lab:\n            if p in ("I", "E") and mode == "strict" and (cur is None or cur[0] != lab):\n                close(); continue\n            close()\n            cur = [lab, s, e]\n        else:\n            cur[2] = e\n        if p in ("E", "S"):\n            close()\n    close()\n    seen, uniq = set(), []\n    for x in out:                                # never emit the same span twice\n        if x not in seen:\n            seen.add(x); uniq.append(x)\n    return uniq\n\n\ndef allowed_transitions(labels, scheme):\n    """(allowed[prev, cur], allowed_start[cur], allowed_end[prev]) as nested lists of bools for a tag scheme."""\n    names = label_names(labels, scheme)\n    C = len(names)\n    pl = [("O", None) if n == "O" else tuple(n.split("-", 1)) for n in names]\n    allow = [[True] * C for _ in range(C)]\n    start, end = [True] * C, [True] * C\n    if scheme == "IO":\n        return allow, start, end\n    for j, (pj, lj) in enumerate(pl):\n        if pj in ("I", "E"):\n            start[j] = False                       # an entity cannot begin with I- / E-\n        for i, (pi, li) in enumerate(pl):\n            if pj in ("I", "E"):                   # I-X / E-X only continue an open X entity\n                ok = li == lj and (pi in ("B", "I"))\n            else:                                  # O, B-, S- may follow anything that has closed\n                ok = pi in ("O", "E", "S") or (scheme == "BIO" and pi in ("B", "I"))\n            allow[i][j] = ok\n        if scheme == "BIOES" and pj in ("B", "I"):\n            end[j] = False                         # a BIOES entity must be closed by E- or S-\n    return allow, start, end\n\n\ndef viterbi_tags(logits, labels, scheme):\n    """Best VALID tag sequence for one document. logits: numpy [n, C] (averaged window logits)."""\n    import numpy as np\n    allow, start, end = allowed_transitions(labels, scheme)\n    n, C = logits.shape\n    if n == 0:\n        return []\n    x = logits.astype(np.float64)\n    lp = x - x.max(1, keepdims=True)\n    lp = lp - np.log(np.exp(lp).sum(1, keepdims=True))      # log-softmax per token\n    NEG = -1e18\n    T = np.where(np.array(allow), 0.0, NEG)                  # [prev, cur]\n    score = lp[0] + np.where(np.array(start), 0.0, NEG)\n    back = np.zeros((n, C), dtype=np.int32)\n    for t in range(1, n):\n        cand = score[:, None] + T                            # [prev, cur]\n        back[t] = cand.argmax(0)\n        score = cand.max(0) + lp[t]\n    score = score + np.where(np.array(end), 0.0, NEG)\n    tags = [int(score.argmax())]\n    for t in range(n - 1, 0, -1):\n        tags.append(int(back[t, tags[-1]]))\n    return tags[::-1]\n\n\n# ------------------------------------------------------------------ windows\ndef make_windows(n_tokens, window, stride):\n    """Start positions so every token is covered; the last window is end-aligned (no padding unless n < window)."""\n    if n_tokens <= window:\n        return [0]\n    starts = list(range(0, n_tokens - window + 1, stride))\n    if starts[-1] + window < n_tokens:\n        starts.append(n_tokens - window)\n    return starts\n\n\n# ------------------------------------------------------------------ scoring\ndef strict_scores(pred, gold, labels):\n    """pred/gold: {doc: [(s, e, label)]} in the SAME convention. Exact match on (doc, s, e, label)."""\n    tp, fp, fn = Counter(), Counter(), Counter()\n    for d in set(gold) | set(pred):\n        g = set(map(tuple, gold.get(d, [])))\n        p = set(map(tuple, pred.get(d, [])))\n        for x in p & g: tp[x[2]] += 1\n        for x in p - g: fp[x[2]] += 1\n        for x in g - p: fn[x[2]] += 1\n    per = {}\n    for l in labels:\n        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0\n        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0\n        F = 2 * P * R / (P + R) if P + R else 0.0\n        per[l] = dict(p=P, r=R, f1=F, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])\n    present = [l for l in labels if per[l]["support"] or per[l]["fp"]]\n    macro = sum(per[l]["f1"] for l in labels) / len(labels)\n    macro_present = sum(per[l]["f1"] for l in present) / max(1, len(present))\n    mp = sum(per[l]["p"] for l in labels) / len(labels)\n    mr = sum(per[l]["r"] for l in labels) / len(labels)\n    return dict(macro_f1=macro, macro_f1_present=macro_present, macro_p=mp, macro_r=mr, per_label=per)\n\n\ndef format_scores(sc):\n    lines = [f"strict macro-F1 {sc[\'macro_f1\']:.4f} (over labels present: {sc[\'macro_f1_present\']:.4f}) | "\n             f"macro-P {sc[\'macro_p\']:.4f} | macro-R {sc[\'macro_r\']:.4f}"]\n    for l, v in sc["per_label"].items():\n        lines.append(f"  {l:10s} F1 {v[\'f1\']:.3f}  P {v[\'p\']:.3f}  R {v[\'r\']:.3f}  "\n                     f"(tp {v[\'tp\']}, fp {v[\'fp\']}, fn {v[\'fn\']}, support {v[\'support\']})")\n    return "\\n".join(lines)\n\n\n# ------------------------------------------------------------------ checks + stats\ndef oracle_check(texts, spans, encode, labels, scheme, mode="lenient"):\n    """Gold spans -> tags -> decoded spans. Score = best achievable with token tags + this tokenizer."""\n    pred, skipped, mid_token = {}, 0, Counter()\n    for d, sp in spans.items():\n        ids, offs = tokenize_doc(texts[d], encode)\n        tags, sk = token_labels(offs, sp, labels, scheme)\n        skipped += sk\n        pred[d] = decode_spans(tags, offs, labels, scheme, mode)\n        starts = {s for s, e in offs if e > s}\n        ends = {e for s, e in offs if e > s}\n        for s, e, lab in sp:\n            if s not in starts or e not in ends:\n                mid_token[lab] += 1\n    sc = strict_scores(pred, spans, labels)\n    return sc, skipped, mid_token\n\n\ndef dataset_stats(texts, spans, encode, labels):\n    lens, n_tok = [], 0\n    for d, t in texts.items():\n        ids, _ = encode(t)\n        lens.append(len(ids)); n_tok += len(ids)\n    lens.sort()\n    cnt = Counter(lab for v in spans.values() for _, _, lab in v) if spans else Counter()\n    overl = 0\n    if spans:\n        for v in spans.values():\n            v = sorted(v)\n            for a, b in zip(v, v[1:]):\n                if b[0] < a[1]:\n                    overl += 1\n    unknown = sorted({lab for v in (spans or {}).values() for _, _, lab in v} - set(labels))\n    return dict(docs=len(texts), tokens=n_tok,\n                doc_tokens_p50=lens[len(lens) // 2] if lens else 0,\n                doc_tokens_p90=lens[int(len(lens) * 0.9)] if lens else 0,\n                doc_tokens_max=lens[-1] if lens else 0,\n                entities=sum(cnt.values()), per_label=dict(cnt.most_common()),\n                overlapping_pairs=overl, unknown_labels=unknown)\n\n\n# ------------------------------------------------------------------ submission\ndef validate_prediction(pred, texts, labels):\n    """pred: {doc: [[s, e, label, (text)]]} in the GOLD convention. Raises on any format problem."""\n    missing = set(texts) - set(pred)\n    extra = set(pred) - set(texts)\n    assert not missing, f"{len(missing)} test docs missing, e.g. {list(missing)[:3]}"\n    assert not extra, f"{len(extra)} unknown doc ids, e.g. {list(extra)[:3]}"\n    n = 0\n    for d, v in pred.items():\n        assert isinstance(v, list), d\n        seen = set()\n        for x in v:\n            assert isinstance(x, list) and len(x) in (3, 4), (d, x)\n            s, e, lab = x[0], x[1], x[2]\n            assert isinstance(s, int) and isinstance(e, int) and 0 <= s <= e <= len(texts[d]), (d, x)\n            assert lab in labels, (d, x)\n            assert (s, e, lab) not in seen, f"duplicate {x} in {d}"\n            seen.add((s, e, lab)); n += 1\n    return n\n')
open("ner_train.py", "w").write('"""\nner_train.py — Phase-2 L-NER fine-tuning harness.\n\n  python ner_train.py config.json\n\nYour file `ner_model.py` (or cfg["model_module"]) provides:\n    build_ner_model(model_cfg: dict, num_labels: int) -> nn.Module\n        - loads your Phase-1 weights itself (paths in model_cfg), adds your head + adaptation\n        - parameters left with requires_grad=True are the ones trained (freeze the rest yourself)\n    forward(input_ids [B,W] long, attention_mask [B,W] bool) -> logits [B,W,num_labels]\n    optional: param_groups(base_lr) -> list of AdamW param-group dicts (e.g. different LR for the head)\n\nThe harness does: tokenization with trimmed offsets, token tags, windows, fp16 training, per-epoch\nvalidation with STRICT span macro-F1, keeps the best weights, predicts the test set, writes\nprediction.json (+ zip) in the gold offset convention, validates it, and optionally:\n  - dumps per-token teacher logits for every training document (knowledge distillation teacher)\n  - trains with a KD loss against such a dump (student)\nEvery number in cfg["train"] / cfg["kd"] / the label scheme / window sizes is YOUR decision.\n"""\nimport importlib, json, math, os, random, sys, time, zipfile\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nimport ner_data as nd\n\nCFG = json.load(open(sys.argv[1]))\nD, TR, MODEL_CFG = CFG["data"], CFG["train"], CFG["model"]\nKD = CFG.get("kd") or {}\nOUT = CFG["out_dir"]; NAME = CFG["run_name"]\nos.makedirs(OUT, exist_ok=True)\nDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nSEED = int(TR.get("seed", 0))\nrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\n\n\ndef log(*a):\n    print(*a, flush=True)\n\n\n# ------------------------------------------------------------------ data\nfrom tokenizers import ByteLevelBPETokenizer\nTOK = ByteLevelBPETokenizer(vocab=os.path.join(D["tokenizer_dir"], "vocab.json"),\n                            merges=os.path.join(D["tokenizer_dir"], "merges.txt"))\nPAD_ID = TOK.token_to_id("<pad>")\nPAD_ID = 0 if PAD_ID is None else PAD_ID\n\n\ndef encode(text):\n    e = TOK.encode(text)\n    return e.ids, e.offsets\n\n\nLABELS = nd.read_labels(D["labels"]) if D.get("labels") else nd.LABELS_DEFAULT\nSCHEME, MODE = D["scheme"], D.get("decode", "lenient")\nTAGS = nd.label_names(LABELS, SCHEME)\nC = len(TAGS)\nW, STRIDE = int(D["window"]), int(D["stride"])\n\ntexts, gold_raw = {}, {}\nfor tp, gp in D["train_files"]:\n    t, g = nd.load_split(tp, gp)\n    texts.update(t); gold_raw.update(g)\nconv = nd.detect_end_convention(texts, gold_raw)\nINCL = conv["inclusive"]\ngold = nd.to_internal(gold_raw, INCL)\ntest_texts = nd.load_split(D["test_file"])[0] if D.get("test_file") else {}\nlog(f"labels {len(LABELS)} | scheme {SCHEME} -> {C} tags | gold end is {\'INCLUSIVE\' if INCL else \'exclusive\'} "\n    f"({conv[\'exclusive_matches\']} exclusive / {conv[\'inclusive_matches\']} inclusive / {conv[\'neither\']} neither)")\n\ndoc_ids = sorted(texts)\nif float(D.get("val_frac", 0)) > 0:\n    tr_ids, va_ids = nd.split_docs(doc_ids, float(D["val_frac"]), int(D.get("split_seed", 0)))\nelse:\n    tr_ids, va_ids = doc_ids, []\nlog(f"train docs {len(tr_ids)} | val docs {len(va_ids)} | test docs {len(test_texts)}")\n\nTOKD = {}                                    # doc -> (ids, trimmed offsets)\n\n\ndef tok(d, src):\n    key = (id(src), d)                       # train and test ids may coincide: cache per source\n    if key not in TOKD:\n        TOKD[key] = nd.tokenize_doc(src[d], encode)\n    return TOKD[key]\n\n\n# ---- teacher logits for KD (per doc, per token)\nTEACH = None\nif KD.get("teacher_logits"):\n    z = np.load(KD["teacher_logits"])\n    meta = json.loads(str(z["__meta__"]))\n    assert meta["tags"] == TAGS, "teacher was trained with a different label scheme"\n    TEACH = {k[2:]: z[k] for k in z.files if k.startswith("d:")}\n    log(f"KD: teacher logits for {len(TEACH)} docs, T={KD[\'temperature\']} alpha={KD[\'alpha\']}")\n\n\ndef build_windows(ids_list, with_tags):\n    """ids_list: doc ids. Returns list of (doc, start, ids[W], tags[W] or None)."""\n    out, skipped = [], 0\n    for d in ids_list:\n        ids, offs = tok(d, texts if with_tags else test_texts)\n        tags = None\n        if with_tags:\n            tags, sk = nd.token_labels(offs, gold.get(d, []), LABELS, SCHEME)\n            skipped += sk\n        for s in nd.make_windows(len(ids), W, STRIDE):\n            out.append((d, s, ids[s:s + W], tags[s:s + W] if tags else None))\n    return out, skipped\n\n\ndef collate(batch, with_tags=True, with_teacher=False):\n    B = len(batch)\n    x = torch.full((B, W), PAD_ID, dtype=torch.long)\n    m = torch.zeros((B, W), dtype=torch.bool)\n    y = torch.full((B, W), -100, dtype=torch.long)\n    tch = torch.zeros((B, W, C), dtype=torch.float32) if (with_teacher and TEACH is not None) else None\n    for i, (d, s, ids, tags) in enumerate(batch):\n        n = len(ids)\n        x[i, :n] = torch.tensor(ids); m[i, :n] = True\n        if with_tags and tags is not None:\n            y[i, :n] = torch.tensor(tags)\n        if tch is not None:\n            tch[i, :n] = torch.from_numpy(TEACH[d][s:s + n].astype(np.float32))\n    return x, m, y, tch\n\n\n# ------------------------------------------------------------------ model\nmodel_lib = importlib.import_module(CFG.get("model_module", "ner_model"))\nmodel = model_lib.build_ner_model(dict(MODEL_CFG), C).to(DEV)\nn_total = sum(p.numel() for p in model.parameters())\nn_train = sum(p.numel() for p in model.parameters() if p.requires_grad)\nlog(f"parameters: total {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M ({100*n_train/max(1,n_total):.2f}%)")\nassert n_train > 0, "no trainable parameters"\n\n# re-score mode: start from a finished run\'s trained parameters (with train.epochs = 0: no training at all,\n# just validation + test prediction with this config\'s decoding)\nif CFG.get("load_trained"):\n    sd = torch.load(CFG["load_trained"], map_location="cpu", weights_only=False)["trained"]\n    cur = dict(model.named_parameters())\n    missing = [k for k in sd if k not in cur]\n    assert not missing, f"load_trained: {len(missing)} names not in this model, e.g. {missing[:3]} (same model dict?)"\n    with torch.no_grad():\n        for k, v in sd.items():\n            cur[k].copy_(v.to(cur[k].device, cur[k].dtype))\n    log(f"loaded {len(sd)} trained tensors from {CFG[\'load_trained\']}")\n\n\n@torch.no_grad()\ndef predict_logits(doc_list, src):\n    """Per-doc token logits averaged over the windows covering each token. -> {doc: np.float32 [n, C]}"""\n    model.eval()\n    out = {}\n    wins = []\n    for d in doc_list:\n        ids, _ = tok(d, src)\n        out[d] = (np.zeros((len(ids), C), np.float32), np.zeros(len(ids), np.float32))\n        for s in nd.make_windows(len(ids), W, STRIDE):\n            wins.append((d, s, ids[s:s + W], None))\n    bs = int(TR.get("eval_batch_size", TR["batch_size"]))\n    for i in range(0, len(wins), bs):\n        b = wins[i:i + bs]\n        x, m, _, _ = collate(b, with_tags=False)\n        with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):\n            lg = model(x.to(DEV), m.to(DEV)).float().cpu().numpy()\n        for j, (d, s, ids, _) in enumerate(b):\n            n = len(ids)\n            out[d][0][s:s + n] += lg[j, :n]; out[d][1][s:s + n] += 1\n    model.train()\n    return {d: a / np.maximum(c, 1)[:, None] for d, (a, c) in out.items()}\n\n\ndef spans_from_logits(lg, src):\n    """MODE \'lenient\' / \'strict\': per-token argmax; \'viterbi\': best valid tag sequence (no I- after O, ...)."""\n    res = {}\n    for d, L in lg.items():\n        _, offs = tok(d, src)\n        if MODE == "viterbi":\n            res[d] = nd.decode_spans(nd.viterbi_tags(L, LABELS, SCHEME), offs, LABELS, SCHEME, "strict")\n        else:\n            res[d] = nd.decode_spans(L.argmax(-1).tolist(), offs, LABELS, SCHEME, MODE)\n    return res\n\n\n# ------------------------------------------------------------------ training\ntrain_w, skipped = build_windows(tr_ids, True)\nlog(f"train windows {len(train_w)} (window {W}, stride {STRIDE}) | overlapping gold entities not representable: {skipped}")\nbs, acc = int(TR["batch_size"]), int(TR.get("grad_accum", 1))\nepochs = int(TR["epochs"])\nsteps_per_epoch = math.ceil(len(train_w) / (bs * acc))\ntotal_steps = steps_per_epoch * epochs\nwarm = int(float(TR.get("warmup_frac", 0)) * total_steps)\nbase_lr = float(TR["lr"])\ngroups = model.param_groups(base_lr) if hasattr(model, "param_groups") else \\\n    [{"params": [p for p in model.parameters() if p.requires_grad]}]\nopt = torch.optim.AdamW(groups, lr=base_lr, weight_decay=float(TR.get("weight_decay", 0.0)))\nfor g in opt.param_groups:\n    g.setdefault("initial_lr", g["lr"])\nscaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")\ncw = TR.get("class_weights")\ncw = torch.tensor(cw, dtype=torch.float32, device=DEV) if cw is not None else None\nif cw is not None:\n    assert len(cw) == C, f"class_weights needs {C} values, one per tag {TAGS}"\n\n\ndef lr_mult(step):\n    if step < warm:\n        return (step + 1) / max(1, warm)\n    sch = TR.get("schedule", "linear")\n    t = (step - warm) / max(1, total_steps - warm)\n    return 1.0 if sch == "constant" else max(0.0, 1 - t) if sch == "linear" else 0.5 * (1 + math.cos(math.pi * t))\n\n\nbest, best_state, history = -1.0, None, []\nif epochs == 0:                                   # re-score only\n    best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}\nstep, t0 = 0, time.time()\nmax_min = float(TR.get("max_minutes", 1e9))\nif DEV.type == "cuda":\n    torch.cuda.reset_peak_memory_stats()\nmodel.train()\nfor ep in range(1, epochs + 1):\n    random.shuffle(train_w)\n    run_loss, nb = 0.0, 0\n    for i in range(0, len(train_w), bs):\n        x, m, y, tch = collate(train_w[i:i + bs], with_teacher=True)\n        x, m, y = x.to(DEV), m.to(DEV), y.to(DEV)\n        with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):\n            lg = model(x, m)\n        lg = lg.float()\n        loss = F.cross_entropy(lg.reshape(-1, C), y.reshape(-1), weight=cw, ignore_index=-100)\n        if tch is not None:\n            T, a = float(KD["temperature"]), float(KD["alpha"])\n            v = (y != -100) if KD.get("only_labelled", True) else m\n            kl = F.kl_div(F.log_softmax(lg[v] / T, -1), F.softmax(tch.to(DEV)[v] / T, -1), reduction="batchmean")\n            loss = (1 - a) * loss + a * T * T * kl\n        scaler.scale(loss / acc).backward()\n        nb += 1; run_loss += loss.item()\n        if nb % acc == 0:\n            for g in opt.param_groups:\n                g["lr"] = g["initial_lr"] * lr_mult(step)\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],\n                                           float(TR.get("grad_clip", 1.0)))\n            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1\n        if (time.time() - t0) / 60 > max_min:\n            break\n    rec = dict(epoch=ep, train_loss=run_loss / max(1, nb), minutes=(time.time() - t0) / 60)\n    if va_ids:\n        vs = nd.strict_scores(spans_from_logits(predict_logits(va_ids, texts), texts),\n                              {d: gold.get(d, []) for d in va_ids}, LABELS)\n        rec.update(val_macro_f1=vs["macro_f1"], val_macro_p=vs["macro_p"], val_macro_r=vs["macro_r"])\n        score = vs["macro_f1"]\n    else:\n        score = ep                                    # no validation: keep the last epoch\n    history.append(rec)\n    log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))\n    if score > best:\n        best = score\n        best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}\n        log(f"  best so far (epoch {ep})")\n    if (time.time() - t0) / 60 > max_min:\n        log(f"time limit {max_min} min reached"); break\ntrain_min = (time.time() - t0) / 60\npeak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEV.type == "cuda" else 0.0\n\n# restore best, save the trained parameters only (rebuild with build_ner_model + load_state_dict(strict=False))\ncur = dict(model.named_parameters())\nwith torch.no_grad():\n    for k, v in best_state.items():\n        cur[k].copy_(v.to(DEV))\ntorch.save(dict(trained=best_state, cfg=CFG, tags=TAGS), os.path.join(OUT, f"{NAME}_trained_params.pt"))\n\nresult = dict(run=NAME, tags=TAGS, scheme=SCHEME, decode=MODE, window=W, stride=STRIDE,\n              params_total=n_total, params_trainable=n_train, train_minutes=train_min, peak_mem_gb=peak_gb,\n              history=history, gold_end_inclusive=INCL)\nif va_ids:\n    t1 = time.time()\n    vl = predict_logits(va_ids, texts)\n    vsp = spans_from_logits(vl, texts)\n    vs = nd.strict_scores(vsp, {d: gold.get(d, []) for d in va_ids}, LABELS)\n    result.update(val=vs, val_infer_seconds=time.time() - t1)\n    log("BEST checkpoint on validation:\\n" + nd.format_scores(vs))\n    json.dump(nd.from_internal(vsp, INCL), open(os.path.join(OUT, f"{NAME}_val_prediction.json"), "w"))\n\n# ---- teacher dump for knowledge distillation (all training docs, best weights)\nif CFG.get("dump_teacher_logits"):\n    tl = predict_logits(doc_ids, texts)\n    np.savez_compressed(os.path.join(OUT, f"{NAME}_teacher_logits.npz"),\n                        __meta__=json.dumps(dict(tags=TAGS, run=NAME, window=W, stride=STRIDE)),\n                        **{f"d:{d}": v.astype(np.float16) for d, v in tl.items()})\n    log(f"teacher logits written for {len(tl)} docs")\n\n# ---- test set -> prediction.json + submission zip\nif test_texts:\n    t1 = time.time()\n    tsp = spans_from_logits(predict_logits(sorted(test_texts), test_texts), test_texts)\n    pred = nd.from_internal(tsp, INCL)\n    n_ent = nd.validate_prediction(pred, test_texts, LABELS)\n    pj = os.path.join(OUT, f"{NAME}_prediction.json")\n    json.dump(pred, open(pj, "w", encoding="utf-8"), ensure_ascii=False)\n    zp = os.path.join(OUT, f"{NAME}_submission.zip")\n    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:\n        zf.write(pj, arcname="prediction.json")\n    result.update(test_entities=n_ent, test_infer_seconds=time.time() - t1)\n    from collections import Counter\n    log(f"test: {n_ent} entities in {len(pred)} docs | per label {dict(Counter(x[2] for v in pred.values() for x in v))}")\n    log(f"wrote {zp}")\njson.dump(result, open(os.path.join(OUT, f"{NAME}_result.json"), "w"), indent=1)\nlog(f"done: {NAME} | best val macro-F1 {best:.4f} | train {train_min:.1f} min | peak {peak_gb:.1f} GB | "\n    f"trainable {n_train/1e6:.3f}M of {n_total/1e6:.2f}M")\n')
import importlib, ner_data as nd; importlib.reload(nd)
print("updated ner_data.py / ner_train.py | decode options: lenient, strict, viterbi")

def rescore(runs, decode="viterbi"):
    cfgs = []
    for run in runs:
        old = json.load(open(f"{R}/{run}/config.json"))
        new = f"{run}_{decode}"
        p = make_cfg(new, model=old["model"], train=dict(old["train"], epochs=0),
                     data=dict(old["data"], decode=decode), module=old["model_module"])
        c = json.load(open(p)); c["load_trained"] = f"{R}/{run}/{run}_trained_params.pt"
        json.dump(c, open(p, "w"), indent=1); cfgs.append(p)
    run_parallel(cfgs)
    for run in runs:
        new = f"{run}_{decode}"
        try:
            a = json.load(open(f"{R}/{run}/{run}_result.json"))["val"]["macro_f1"]
            b = json.load(open(f"{R}/{new}/{new}_result.json"))["val"]["macro_f1"]
            print(f"{run:18s} val macro-F1 {a:.4f} (original) -> {b:.4f} ({decode}) | "
                  f"zip: {R}/{new}/{new}_submission.zip")
        except Exception as e:
            print(f"{run}: FAILED ({e}) — read {R}/{new}/train.log")

updated ner_data.py / ner_train.py | decode options: lenient, strict, viterbi


In [34]:
rescore(["prefixlm_lorma", "encdec_lora", "biscale_full", "arch4_adapter"])

started /kaggle/working/p2runs/prefixlm_lorma_viterbi/config.json on GPU 0
started /kaggle/working/p2runs/encdec_lora_viterbi/config.json on GPU 1

==== /kaggle/working/p2runs/encdec_lora_viterbi/config.json exit 0 (0.3 min)
labels 12 | scheme BIO -> 25 tags | gold end is exclusive (8393 exclusive / 0 inclusive / 0 neither)
train docs 63 | val docs 7 | test docs 35
parameters: total 68.30M | trainable 1.199M (1.76%)
loaded 50 trained tensors from /kaggle/working/p2runs/encdec_lora/encdec_lora_trained_params.pt
train windows 1101 (window 512, stride 384) | overlapping gold entities not representable: 0
BEST checkpoint on validation:
strict macro-F1 0.1828 (over labels present: 0.1828) | macro-P 0.2221 | macro-R 0.1902
  APP        F1 0.053  P 0.500  R 0.028  (tp 1, fp 1, fn 35, support 36)
  RESP       F1 0.000  P 0.000  R 0.000  (tp 0, fp 1, fn 31, support 31)
  A.COUNSEL  F1 0.000  P 0.000  R 0.000  (tp 0, fp 0, fn 16, support 16)
  R.COUNSEL  F1 0.000  P 0.000  R 0.000  (tp 0, fp 0, 

In [35]:
   from IPython.display import FileLink, display
   import zipfile
   for r in ["biscale_full_viterbi", "arch4_adapter_viterbi", "prefixlm_lorma_viterbi"]:
       z = f"p2runs/{r}/{r}_submission.zip"
       print(r, "| contains:", zipfile.ZipFile(z).namelist())
       display(FileLink(z))

biscale_full_viterbi | contains: ['prediction.json']


/kaggle/working/p2runs/biscale_full_viterbi/biscale_full_viterbi_submission.zip

arch4_adapter_viterbi | contains: ['prediction.json']


/kaggle/working/p2runs/arch4_adapter_viterbi/arch4_adapter_viterbi_submission.zip

prefixlm_lorma_viterbi | contains: ['prediction.json']


/kaggle/working/p2runs/prefixlm_lorma_viterbi/prefixlm_lorma_viterbi_submission.zip

In [36]:
   DATA["decode"] = "viterbi"
   run_round(2)

started /kaggle/working/p2runs/prefixlm_bitfit/config.json on GPU 0
started /kaggle/working/p2runs/kd_student/config.json on GPU 1

==== /kaggle/working/p2runs/kd_student/config.json exit 0 (5.7 min)
epoch 7 | train_loss 0.0260 | minutes 3.5894 | val_macro_f1 0.5076 | val_macro_p 0.5284 | val_macro_r 0.5020
  best so far (epoch 7)
epoch 8 | train_loss 0.0160 | minutes 4.1196 | val_macro_f1 0.5076 | val_macro_p 0.5207 | val_macro_r 0.5087
  best so far (epoch 8)
epoch 9 | train_loss 0.0107 | minutes 4.6512 | val_macro_f1 0.5083 | val_macro_p 0.5213 | val_macro_r 0.5057
  best so far (epoch 9)
epoch 10 | train_loss 0.0090 | minutes 5.1861 | val_macro_f1 0.5123 | val_macro_p 0.5309 | val_macro_r 0.5071
  best so far (epoch 10)
BEST checkpoint on validation:
strict macro-F1 0.5123 (over labels present: 0.5123) | macro-P 0.5309 | macro-R 0.5071
  APP        F1 0.000  P 0.000  R 0.000  (tp 0, fp 18, fn 36, support 36)
  RESP       F1 0.453  P 0.545  R 0.387  (tp 12, fp 10, fn 19, support 31)

In [38]:
   display(FileLink("p2runs/kd_student/kd_student_submission.zip"))

/kaggle/working/p2runs/kd_student/kd_student_submission.zip

In [39]:
display(FileLink("p2runs/prefixlm_bitfit/prefixlm_bitfit_submission.zip"))
display(FileLink("p2runs/kd_student/kd_student_submission.zip"))

/kaggle/working/p2runs/prefixlm_bitfit/prefixlm_bitfit_submission.zip

/kaggle/working/p2runs/kd_student/kd_student_submission.zip

In [40]:
DATA_V = dict(DATA, decode="viterbi")
i1 = make_cfg("pfx_try1", model=dict(phase1="prefixlm", method=..., ...your keys...),
              train=T(...your changes...), data=dict(DATA_V, ...e.g. window/scheme changes...))
i2 = make_cfg("pfx_try2", model=dict(phase1="prefixlm", method=..., ...), train=T(...), data=DATA_V)
run_parallel([i1, i2])

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1845952569.py, line 2)

In [41]:
import zipfile, glob, os
from IPython.display import FileLink
keep = [p for p in glob.glob("p2runs/**/*", recursive=True)
        if p.endswith(("config.json", "_result.json", "train.log", "_val_prediction.json", "results.csv"))]
with zipfile.ZipFile("phase2_report_files.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in keep: z.write(p)
print(len(keep), "files"); display(FileLink("phase2_report_files.zip"))

48 files


/kaggle/working/phase2_report_files.zip

In [42]:
# 10 — results table (for the report) + pick a zip to submit
import pandas as pd
rows = []
for rp in sorted(glob.glob("/kaggle/working/p2runs/*/*_result.json")):
    r = json.load(open(rp))
    if r["run"].startswith("smoke"): continue
    v = r.get("val", {})
    rows.append(dict(run=r["run"], val_macro_f1=v.get("macro_f1"), val_P=v.get("macro_p"), val_R=v.get("macro_r"),
                     trainable_M=r["params_trainable"] / 1e6, total_M=r["params_total"] / 1e6,
                     train_min=r["train_minutes"], peak_GB=r["peak_mem_gb"], scheme=r["scheme"], window=r["window"],
                     test_entities=r.get("test_entities")))
df = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False); display(df)
df.to_csv("/kaggle/working/p2runs/results.csv", index=False)
SUBMIT = None
if SUBMIT:
    shutil.copy(f"/kaggle/working/p2runs/{SUBMIT}/{SUBMIT}_submission.zip", "/kaggle/working/submission.zip")
    print("submission.zip =", SUBMIT)

,run,val_macro_f1,val_P,val_R,trainable_M,total_M,train_min,peak_GB,scheme,window,test_entities
9,prefixlm_lorma_viterbi,0.532819,0.551324,0.530260,2.673433,112.305433,0.000025,0.450009,BIO,512,4087
6,kd_student,0.512270,0.530874,0.507129,89.698585,89.698585,5.214323,3.198387,BIO,512,4130
7,prefixlm_bitfit,0.507925,0.523161,0.509279,0.121369,109.651225,7.321793,1.818363,BIO,512,3941
3,biscale_full_viterbi,0.494211,0.509310,0.488701,149.491993,149.491993,0.000020,0.685271,BIO,512,4159
1,arch4_adapter_viterbi,0.464845,0.490171,0.457370,7.112857,116.714905,0.000011,0.471845,BIO,512,3968
8,prefixlm_lorma,0.437651,0.406415,0.486207,2.673433,112.305433,9.337709,2.525868,BIO,512,4881
2,biscale_full,0.432270,0.414152,0.464810,149.491993,149.491993,10.458255,5.898372,BIO,512,4922
0,arch4_adapter,0.364214,0.342162,0.400243,7.112857,116.714905,7.230703,2.657880,BIO,512,4929
5,encdec_lora_viterbi,0.182811,0.222144,0.190214,1.198873,68.303641,0.000010,0.274002,BIO,512,3411
4,encdec_lora,0.123621,0.121731,0.157758,1.198873,68.303641,3.935388,1.374430,BIO,512,6655


In [43]:
import zipfile, os
from IPython.display import FileLink
runs = ["prefixlm_lorma", "kd_student", "biscale_full", "arch4_adapter", "prefixlm_bitfit", "encdec_lora"]
with zipfile.ZipFile("phase2_weights.zip", "w", zipfile.ZIP_STORED) as z:
    for r in runs:
        for f in (f"p2runs/{r}/{r}_trained_params.pt", f"p2runs/{r}/config.json"):
            if os.path.exists(f): z.write(f)
for r in runs:
    f = f"p2runs/{r}/{r}_trained_params.pt"
    if os.path.exists(f): print(f"{r:16s} {os.path.getsize(f)/1e6:7.1f} MB")
display(FileLink("phase2_weights.zip"))

prefixlm_lorma      10.7 MB
kd_student         358.8 MB
biscale_full       598.0 MB
arch4_adapter       28.5 MB
prefixlm_bitfit      0.5 MB
encdec_lora          4.8 MB


/kaggle/working/phase2_weights.zip